In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Environment Setup
# ════════════════════════════════════════════════════════════════
import subprocess, sys

# Install / upgrade required packages
pkgs = [
    'torch>=2.0.0',
    'scikit-learn>=1.3.0',
    'numpy>=1.24.0',
    'pandas>=2.0.0',
    'matplotlib>=3.7.0',
    'tqdm>=4.65.0',
    'joblib>=1.3.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Packages ready')

Packages ready


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Imports & GPU Check
# ════════════════════════════════════════════════════════════════
import os, re, math, time, json, random, collections, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    cohen_kappa_score, log_loss, hamming_loss, jaccard_score,
    balanced_accuracy_score, confusion_matrix, roc_curve,
    precision_recall_curve,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')

Device: cpu


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Directory Setup & Seeding
# ════════════════════════════════════════════════════════════════
BASE_DIR   = '/content/hdfs_anomaly'
DATA_DIR   = f'{BASE_DIR}/data'
MODEL_DIR  = f'{BASE_DIR}/models'
RESULTS_DIR= f'{BASE_DIR}/results'

for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Global hyperparameters ────────────────────────────────────
CFG = dict(
    VOCAB_SIZE    = 75,
    EMBED_DIM     = 64,
    NUM_HEADS     = 4,
    NUM_LAYERS    = 2,
    FF_DIM        = 128,
    DROPOUT       = 0.1,
    MAX_SEQ_LEN   = 100,
    BATCH_SIZE    = 64,
    LEARNING_RATE = 1e-3,
    WEIGHT_DECAY  = 1e-4,
    NUM_EPOCHS    = 30,
    PATIENCE      = 5,
    GRAD_CLIP     = 1.0,
    USE_AMP       = True,
    TRAIN_RATIO   = 0.70,
    VAL_RATIO     = 0.15,
    RANDOM_SEED   = 42,
    RF_CLASS_WEIGHT='balanced',
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CFG['RANDOM_SEED'])
print('Directories created. Seed set.')

Directories created. Seed set.


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Vocabulary & Tokenization
# ════════════════════════════════════════════════════════════════

PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
CLS_TOKEN = '<CLS>'

class Vocabulary:
    def __init__(self):
        self.token2id = {}
        self.id2token = {}
        self._counter = collections.Counter()
        self.pad_id = 0
        self.unk_id = 1
        self.cls_id = 2

    def build(self, sequences, min_freq=1):
        for seq in sequences: self._counter.update(seq)
        self.token2id = {PAD_TOKEN: 0, UNK_TOKEN: 1, CLS_TOKEN: 2}
        idx = 3
        for tok, cnt in self._counter.most_common():
            if cnt >= min_freq:
                self.token2id[tok] = idx; idx += 1
        self.id2token = {v: k for k, v in self.token2id.items()}
        return self

    def encode(self, seq):
        return [self.token2id.get(t, self.unk_id) for t in seq]

    def __len__(self): return len(self.token2id)

def encode_and_pad(seq, vocab, max_len):
    ids = [vocab.cls_id] + vocab.encode(seq)
    ids = ids[:max_len]
    ids += [vocab.pad_id] * (max_len - len(ids))
    return ids

# ── PyTorch Dataset ───────────────────────────────────────────
class HDFSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def get_loaders(splits):
    loaders = {}
    for split, (X, y) in splits.items():
        loaders[split] = DataLoader(
            HDFSDataset(X, y), batch_size=CFG['BATCH_SIZE'],
            shuffle=(split == 'train'), num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )
    return loaders

print('Vocabulary & Dataset classes ready.')

Vocabulary & Dataset classes ready.


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Install + Download Dataset from Kaggle
# ════════════════════════════════════════════════════════════════

!pip install -q kagglehub

import kagglehub
import os

# Download dataset
DATA_DIR = kagglehub.dataset_download(
    "ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data"
)

print("Dataset Path:", DATA_DIR)

print("\nFiles Inside Dataset:")
print(os.listdir(DATA_DIR))

100%|██████████| 1.55G/1.55G [00:24<00:00, 69.0MB/s]

Extracting files...


Dataset Path: /root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3

Files Inside Dataset:
['HDFS_v2', 'HDFS_v1', 'HDFS_v3_TraceBench', 'HDFS_2k']


In [ ]:
TRACEBENCH_DIR = '/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v3_TraceBench/tracebench'

import os

print(TRACEBENCH_DIR)

print(os.listdir(TRACEBENCH_DIR)[:10])

/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v3_TraceBench/tracebench
['AN_Data_cutMeta_r_10FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Net_slowHDFS_rpc_0400ms_30C_0to0INT_5RT_1WT', 'NM_DN_rw_30DN_30C_1to19B_0to120INTR_0to600INTW_50RT_10WT', 'AN_Proc_killDN_r_03FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Net_slowHDFS_w_18ms_30C_1to19B_0to600INT_15RT_5WT', 'AN_Data_lossMeta_r_40FDN_30C_1to19B_0to120INT_15RT_5WT', 'NM_CL_rpc_35C_0to0INT_10RT_1WT', 'COM_Sin_Proc_rwrpc_3_30C_1to19B_0to120INTR_0to600INTW_0to0INTRPC_60RT_10WT', 'AN_Sys_panicDN_r_05FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Proc_suspendDN_r_50FDN_30C_1to19B_0to120INT_15RT_10WT']


In [ ]:
import os

print(os.listdir(TRACEBENCH_DIR)[:10])

['AN_Data_cutMeta_r_10FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Net_slowHDFS_rpc_0400ms_30C_0to0INT_5RT_1WT', 'NM_DN_rw_30DN_30C_1to19B_0to120INTR_0to600INTW_50RT_10WT', 'AN_Proc_killDN_r_03FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Net_slowHDFS_w_18ms_30C_1to19B_0to600INT_15RT_5WT', 'AN_Data_lossMeta_r_40FDN_30C_1to19B_0to120INT_15RT_5WT', 'NM_CL_rpc_35C_0to0INT_10RT_1WT', 'COM_Sin_Proc_rwrpc_3_30C_1to19B_0to120INTR_0to600INTW_0to0INTRPC_60RT_10WT', 'AN_Sys_panicDN_r_05FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Proc_suspendDN_r_50FDN_30C_1to19B_0to120INT_15RT_10WT']


In [ ]:
# ════════════════════════════════════════════════════════════════
# FIND ALL FILES RECURSIVELY
# ════════════════════════════════════════════════════════════════

import os

for root, dirs, files in os.walk(DATA_DIR):

    for file in files:

        print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/README.md
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-11.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-secondarynamenode-mesos-01.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-26.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-07.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-03.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v

In [ ]:
STRUCTURED_CSV = (
    "/kaggle/input/"
    "loghub-hdfs-hadoop-distributed-file-system-data/"
    "HDFS_2k/HDFS_2k.log_structured.csv"
)

loaders, splits = prepare_real_data(
    STRUCTURED_CSV
)

NameError: name 'prepare_real_data' is not defined

In [ ]:
import os

for root, dirs, files in os.walk(DATA_DIR):

    for file in files:

        if '.log' in file or '.csv' in file:

            print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-11.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-secondarynamenode-mesos-01.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-26.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-07.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-03.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v2/node_logs/hadoop-hdfs-datanode-mesos-08.log
/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-

In [ ]:
import os

TRACEBENCH_ROOT = (
    "/kaggle/input/"
    "loghub-hdfs-hadoop-distributed-file-system-data/"
    "HDFS_v3_TraceBench/tracebench"
)

count = 0

for root, dirs, files in os.walk(TRACEBENCH_ROOT):

    for f in files:

        if f == "event.csv":

            print(os.path.join(root, f))

            count += 1

            if count == 3:
                break

    if count == 3:
        break

print("FOUND:", count)

FOUND: 0


In [ ]:
import pandas as pd

path = "/kaggle/input/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_v3_TraceBench/tracebench/AN_Proc_suspendDN_w_35FDN_30C_1to19B_0to600INT_15RT_5WT/event.csv"

df = pd.read_csv(path)

print("COLUMNS:\n")
print(df.columns)

print("\nHEAD:\n")
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_v3_TraceBench/tracebench/AN_Proc_suspendDN_w_35FDN_30C_1to19B_0to600INT_15RT_5WT/event.csv'

In [ ]:
import pandas as pd

path = "/kaggle/input/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_v3_TraceBench/tracebench/AN_Proc_suspendDN_w_35FDN_30C_1to19B_0to600INT_15RT_5WT/event.csv"

df = pd.read_csv(path)

print(df['OpName'].head(20))

print("\nUnique Operations:\n")
print(df['OpName'].nunique())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_v3_TraceBench/tracebench/AN_Proc_suspendDN_w_35FDN_30C_1to19B_0to600INT_15RT_5WT/event.csv'

In [ ]:
path = "/kaggle/input/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_v3_TraceBench/tracebench/AN_Proc_suspendDN_w_35FDN_30C_1to19B_0to600INT_15RT_5WT/event.csv"

with open(path, 'r') as f:

    for i in range(10):

        print(f.readline())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_v3_TraceBench/tracebench/AN_Proc_suspendDN_w_35FDN_30C_1to19B_0to600INT_15RT_5WT/event.csv'

In [ ]:
# ════════════════════════════════════════════════════════════════
# ACTUALLY CORRECT TRACEBENCH LOADER
# ════════════════════════════════════════════════════════════════

import os
import csv
import numpy as np

from sklearn.model_selection import train_test_split


TRACEBENCH_ROOT = (
    "/kaggle/input/"
    "loghub-hdfs-hadoop-distributed-file-system-data/"
    "HDFS_v3_TraceBench/tracebench"
)

CFG = {
    'MAX_SEQ_LEN': 100,
    'TRAIN_RATIO': 0.7,
    'VAL_RATIO': 0.15,
    'RANDOM_SEED': 42,
}


all_sequences = []
all_labels = []

event_vocab = set()

scenario_count = 0


# ------------------------------------------------
# LOAD DATA
# ------------------------------------------------

for scenario_folder in os.listdir(TRACEBENCH_ROOT):

    scenario_path = os.path.join(
        TRACEBENCH_ROOT,
        scenario_folder
    )

    if not os.path.isdir(scenario_path):
        continue

    label = 1 if scenario_folder.startswith('AN_') else 0

    event_csv = os.path.join(
        scenario_path,
        'event.csv'
    )

    if not os.path.exists(event_csv):
        continue

    sequence = []

    try:

        with open(event_csv, 'r', encoding='utf-8') as f:

            reader = csv.reader(f)

            # skip header
            header = next(reader)

            # Find OpName index
            op_idx = header.index('OpName')

            for row in reader:

                try:

                    op = row[op_idx].strip()

                    if len(op) == 0:
                        continue

                    sequence.append(op)

                    event_vocab.add(op)

                except:
                    continue

    except Exception as e:

        print("Skipping:", scenario_folder)

        continue

    if len(sequence) == 0:
        continue

    all_sequences.append(sequence)

    all_labels.append(label)

    scenario_count += 1


# ------------------------------------------------
# SUMMARY
# ------------------------------------------------

print("Loaded Scenarios:", scenario_count)

print("Vocabulary Size:", len(event_vocab))


# ------------------------------------------------
# VOCAB
# ------------------------------------------------

event2id = {
    ev: idx + 1
    for idx, ev in enumerate(sorted(event_vocab))
}

VOCAB_SIZE = len(event2id) + 1

print("Final Vocabulary Size:", VOCAB_SIZE)


# ------------------------------------------------
# ENCODE
# ------------------------------------------------

X = []

for seq in all_sequences:

    encoded = [
        event2id[e]
        for e in seq
    ]

    encoded = encoded[:CFG['MAX_SEQ_LEN']]

    if len(encoded) < CFG['MAX_SEQ_LEN']:

        encoded += [0] * (
            CFG['MAX_SEQ_LEN'] - len(encoded)
        )

    X.append(encoded)

X = np.array(X, dtype=np.int64)

y = np.array(all_labels, dtype=np.int64)


print("\nDataset Shape:")
print(X.shape)

print("\nLabels Shape:")
print(y.shape)


# ------------------------------------------------
# SPLIT
# ------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=(1 - CFG['TRAIN_RATIO']),
    stratify=y,
    random_state=CFG['RANDOM_SEED']
)

val_ratio_adjusted = (
    CFG['VAL_RATIO']
    /
    (1 - CFG['TRAIN_RATIO'])
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=(1 - val_ratio_adjusted),
    stratify=y_temp,
    random_state=CFG['RANDOM_SEED']
)

print("\nDataset Statistics:\n")

for name, labels in [
    ('train', y_train),
    ('val', y_val),
    ('test', y_test)
]:

    pos = labels.sum()

    print(
        f'{name}: {len(labels)} | '
        f'anomaly={pos} '
        f'({pos/len(labels)*100:.2f}%)'
    )


# ------------------------------------------------
# EXAMPLES
# ------------------------------------------------

print("\nExample Raw Sequence:\n")
print(all_sequences[0][:20])

print("\nExample Encoded Sequence:\n")
print(X_train[0][:20])

Loaded Scenarios: 364
Vocabulary Size: 74
Final Vocabulary Size: 75

Dataset Shape:
(364, 100)

Labels Shape:
(364,)

Dataset Statistics:

train: 254 | anomaly=180 (70.87%)
val: 54 | anomaly=38 (70.37%)
test: 56 | anomaly=40 (71.43%)

Example Raw Sequence:

['getFileInfo', 'RPC:getFileInfo', 'create', 'RPC:create', 'addBlock', 'RPC:addBlock', 'getFileInfo', 'RPC:getFileInfo', 'create', 'RPC:create', 'addBlock', 'RPC:addBlock', 'getFileInfo', 'RPC:getFileInfo', 'create', 'RPC:create', 'addBlock', 'RPC:addBlock', 'getFileInfo', 'RPC:getFileInfo']

Example Encoded Sequence:

[56 56 18 18 59 21 59 48 21 48 56 18 56 59 21 48 18 59 21 48]


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7 — Lightweight Transformer Encoder
# Optimized for TraceBench Hybrid Pipeline
# ════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=512, dropout=0.1):

        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)

        pos = torch.arange(
            0,
            max_len,
            dtype=torch.float
        ).unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div)

        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer(
            'pe',
            pe.unsqueeze(0)
        )

    def forward(self, x):

        x = x + self.pe[:, :x.size(1)]

        return self.dropout(x)


# ════════════════════════════════════════════════════════════════
# LIGHT TRANSFORMER
# ════════════════════════════════════════════════════════════════

class LightTransformerEncoder(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        D = CFG['EMBED_DIM']

        # ------------------------------------------------
        # Embedding
        # ------------------------------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            D,
            padding_idx=0
        )

        # ------------------------------------------------
        # Positional Encoding
        # ------------------------------------------------

        self.pos_enc = PositionalEncoding(
            D,
            CFG['MAX_SEQ_LEN'],
            CFG['DROPOUT']
        )

        # ------------------------------------------------
        # Transformer Layer
        # ------------------------------------------------

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D,
            nhead=CFG['NUM_HEADS'],
            dim_feedforward=CFG['FF_DIM'],
            dropout=CFG['DROPOUT'],
            batch_first=True,
            norm_first=True,
            activation='gelu'
        )

        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=CFG['NUM_LAYERS']
        )

        # ------------------------------------------------
        # Layer Norm
        # ------------------------------------------------

        self.norm = nn.LayerNorm(D)

        # ------------------------------------------------
        # Projection Head
        # (small because hybrid model handles fusion later)
        # ------------------------------------------------

        self.proj = nn.Sequential(

            nn.Linear(D, D),

            nn.GELU(),

            nn.Dropout(CFG['DROPOUT'])
        )

        self._init_weights()

    # ------------------------------------------------
    # Weight Initialization
    # ------------------------------------------------

    def _init_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Linear):

                nn.init.xavier_uniform_(m.weight)

                if m.bias is not None:

                    nn.init.zeros_(m.bias)

            elif isinstance(m, nn.Embedding):

                nn.init.normal_(
                    m.weight,
                    mean=0,
                    std=0.02
                )

    # ------------------------------------------------
    # Mean Pooling
    # ------------------------------------------------

    def _pool(self, enc, pad_mask):

        mask = (~pad_mask).unsqueeze(-1).float()

        pooled = (
            (enc * mask).sum(dim=1)
            /
            mask.sum(dim=1).clamp(min=1e-9)
        )

        return pooled

    # ------------------------------------------------
    # Forward
    # ------------------------------------------------

    def forward(self, x, return_emb=False):

        pad_mask = (x == 0)

        tok = self.embedding(x)

        tok = self.pos_enc(tok)

        enc = self.encoder(
            tok,
            src_key_padding_mask=pad_mask
        )

        enc = self.norm(enc)

        pooled = self._pool(enc, pad_mask)

        emb = self.proj(pooled)

        if return_emb:

            return emb

        return emb

    # ------------------------------------------------
    # Extract Embeddings
    # ------------------------------------------------

    def extract_embeddings(self, x):

        return self.forward(
            x,
            return_emb=True
        )


# ════════════════════════════════════════════════════════════════
# INITIALIZE MODEL
# ════════════════════════════════════════════════════════════════

model = LightTransformerEncoder(
    vocab_size=VOCAB_SIZE
).to(DEVICE)

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f'Model parameters: {n_params:,}')

print(f'Device: {DEVICE}')

Model parameters: 76,032
Device: cpu


In [ ]:
# ════════════════════════════════════════════════════════════════
# CREATE PYTORCH DATALOADERS
# ════════════════════════════════════════════════════════════════

from torch.utils.data import TensorDataset, DataLoader


# ------------------------------------------------
# DATASETS
# ------------------------------------------------

train_dataset = TensorDataset(

    torch.tensor(
        X_train,
        dtype=torch.long
    ),

    torch.tensor(
        y_train,
        dtype=torch.long
    )
)

val_dataset = TensorDataset(

    torch.tensor(
        X_val,
        dtype=torch.long
    ),

    torch.tensor(
        y_val,
        dtype=torch.long
    )
)

test_dataset = TensorDataset(

    torch.tensor(
        X_test,
        dtype=torch.long
    ),

    torch.tensor(
        y_test,
        dtype=torch.long
    )
)


# ------------------------------------------------
# DATALOADERS
# ------------------------------------------------

loaders = {

    'train': DataLoader(

        train_dataset,

        batch_size=CFG['BATCH_SIZE'],

        shuffle=True,

        num_workers=2,

        pin_memory=True
    ),

    'val': DataLoader(

        val_dataset,

        batch_size=CFG['BATCH_SIZE'],

        shuffle=False,

        num_workers=2,

        pin_memory=True
    ),

    'test': DataLoader(

        test_dataset,

        batch_size=CFG['BATCH_SIZE'],

        shuffle=False,

        num_workers=2,

        pin_memory=True
    )
}


print("DataLoaders created successfully.")

print("Train batches:", len(loaders['train']))

print("Validation batches:", len(loaders['val']))

print("Test batches:", len(loaders['test']))

DataLoaders created successfully.
Train batches: 4
Validation batches: 1
Test batches: 1


In [ ]:
# ════════════════════════════════════════════════════════════════
# FIXED TRANSFORMER MODEL
# Correct Positional Encoding Size
# ════════════════════════════════════════════════════════════════

import math
import torch
import torch.nn as nn


# IMPORTANT FIX
CFG['MAX_SEQ_LEN'] = 100


# ════════════════════════════════════════════════════════════════
# POSITIONAL ENCODING
# ════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len,
        dropout=0.1
    ):

        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):

        seq_len = x.size(1)

        x = x + self.pe[:, :seq_len]

        return self.dropout(x)


# ════════════════════════════════════════════════════════════════
# LIGHT TRANSFORMER ENCODER
# ════════════════════════════════════════════════════════════════

class LightTransformerEncoder(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        D = CFG['EMBED_DIM']

        # ------------------------------------------------
        # Embedding
        # ------------------------------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            D,
            padding_idx=0
        )

        # ------------------------------------------------
        # FIXED POSITIONAL ENCODING
        # ------------------------------------------------

        self.pos_enc = PositionalEncoding(
            d_model=D,
            max_len=CFG['MAX_SEQ_LEN'],
            dropout=CFG['DROPOUT']
        )

        # ------------------------------------------------
        # Transformer
        # ------------------------------------------------

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D,
            nhead=CFG['NUM_HEADS'],
            dim_feedforward=CFG['FF_DIM'],
            dropout=CFG['DROPOUT'],
            batch_first=True,
            norm_first=True,
            activation='gelu'
        )

        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=CFG['NUM_LAYERS']
        )

        self.norm = nn.LayerNorm(D)

        # ------------------------------------------------
        # Projection Head
        # ------------------------------------------------

        self.proj = nn.Sequential(

            nn.Linear(D, D),

            nn.GELU(),

            nn.Dropout(CFG['DROPOUT'])
        )

        self._init_weights()

    # ------------------------------------------------
    # Initialization
    # ------------------------------------------------

    def _init_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Linear):

                nn.init.xavier_uniform_(m.weight)

                if m.bias is not None:

                    nn.init.zeros_(m.bias)

            elif isinstance(m, nn.Embedding):

                nn.init.normal_(
                    m.weight,
                    mean=0,
                    std=0.02
                )

    # ------------------------------------------------
    # Mean Pooling
    # ------------------------------------------------

    def _pool(self, enc, pad_mask):

        mask = (~pad_mask).unsqueeze(-1).float()

        pooled = (
            (enc * mask).sum(dim=1)
            /
            mask.sum(dim=1).clamp(min=1e-9)
        )

        return pooled

    # ------------------------------------------------
    # Forward
    # ------------------------------------------------

    def forward(self, x):

        pad_mask = (x == 0)

        tok = self.embedding(x)

        tok = self.pos_enc(tok)

        enc = self.encoder(
            tok,
            src_key_padding_mask=pad_mask
        )

        enc = self.norm(enc)

        pooled = self._pool(
            enc,
            pad_mask
        )

        emb = self.proj(pooled)

        return emb


# ════════════════════════════════════════════════════════════════
# BUILD MODEL
# ════════════════════════════════════════════════════════════════

model = LightTransformerEncoder(
    vocab_size=VOCAB_SIZE
).to(DEVICE)


# ════════════════════════════════════════════════════════════════
# VERIFY POSITIONAL ENCODING
# ════════════════════════════════════════════════════════════════

print(
    "Positional Encoding Shape:",
    model.pos_enc.pe.shape
)

print(
    "Expected Sequence Length:",
    CFG['MAX_SEQ_LEN']
)


# ════════════════════════════════════════════════════════════════
# PARAMS
# ════════════════════════════════════════════════════════════════

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f'\nModel parameters: {n_params:,}')

print(f'Device: {DEVICE}')

Positional Encoding Shape: torch.Size([1, 100, 64])
Expected Sequence Length: 100

Model parameters: 76,032
Device: cpu


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8 — FIXED TRANSFORMER TRAINING
# + TEST LOSS / TEST ACC
# ════════════════════════════════════════════════════════════════

def compute_class_weights(y):

    counts = np.bincount(
        y,
        minlength=2
    ).astype(float)

    w = counts.sum() / (
        2.0 * counts + 1e-9
    )

    return torch.tensor(
        w,
        dtype=torch.float32
    ).to(DEVICE)


# ════════════════════════════════════════════════════════════════
# RUN EPOCH
# ════════════════════════════════════════════════════════════════

def run_epoch(
    model,
    classifier_head,
    loader,
    criterion,
    optimizer=None,
    scaler=None,
    train=True
):

    model.train() if train else model.eval()

    classifier_head.train() if train else classifier_head.eval()

    total_loss = 0

    correct = 0

    total = 0

    ctx = (
        torch.enable_grad()
        if train else
        torch.no_grad()
    )

    with ctx:

        for X, y in loader:

            X = X.to(DEVICE)

            y = y.to(DEVICE)

            if train:

                optimizer.zero_grad(
                    set_to_none=True
                )

            use_amp = (
                CFG['USE_AMP']
                and
                torch.cuda.is_available()
            )

            # ------------------------------------------------
            # AMP
            # ------------------------------------------------

            if use_amp:

                with autocast():

                    embs = model(X)

                    logits = classifier_head(embs)

                    loss = criterion(
                        logits,
                        y
                    )

                if train:

                    scaler.scale(loss).backward()

                    scaler.unscale_(optimizer)

                    nn.utils.clip_grad_norm_(
                        model.parameters(),
                        CFG['GRAD_CLIP']
                    )

                    scaler.step(optimizer)

                    scaler.update()

            # ------------------------------------------------
            # FP32
            # ------------------------------------------------

            else:

                embs = model(X)

                logits = classifier_head(embs)

                loss = criterion(
                    logits,
                    y
                )

                if train:

                    loss.backward()

                    nn.utils.clip_grad_norm_(
                        model.parameters(),
                        CFG['GRAD_CLIP']
                    )

                    optimizer.step()

            total_loss += (
                loss.item()
                *
                y.size(0)
            )

            correct += (
                logits.argmax(1) == y
            ).sum().item()

            total += y.size(0)

    return (

        total_loss / total,

        correct / total
    )


# ════════════════════════════════════════════════════════════════
# CLASSIFIER HEAD
# ════════════════════════════════════════════════════════════════

classifier_head = nn.Sequential(

    nn.Linear(
        CFG['EMBED_DIM'],
        CFG['FF_DIM'] // 2
    ),

    nn.GELU(),

    nn.Dropout(
        CFG['DROPOUT']
    ),

    nn.Linear(
        CFG['FF_DIM'] // 2,
        2
    )

).to(DEVICE)


# ════════════════════════════════════════════════════════════════
# TRAINING SETUP
# ════════════════════════════════════════════════════════════════

set_seed(
    CFG['RANDOM_SEED']
)

class_w = compute_class_weights(
    splits['train'][1]
)

criterion = nn.CrossEntropyLoss(
    weight=class_w
)

optimizer = optim.AdamW(

    list(model.parameters())
    +
    list(classifier_head.parameters()),

    lr=CFG['LEARNING_RATE'],

    weight_decay=CFG['WEIGHT_DECAY']
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=CFG['NUM_EPOCHS'],

    eta_min=CFG['LEARNING_RATE'] * 0.01
)

scaler = (

    GradScaler()

    if (

        CFG['USE_AMP']

        and

        torch.cuda.is_available()

    )

    else None
)


# ════════════════════════════════════════════════════════════════
# TRAIN LOOP
# ════════════════════════════════════════════════════════════════

history = collections.defaultdict(list)

best_val = float('inf')

patience_c = 0

CKPT_PATH = (
    f'{MODEL_DIR}/transformer_best.pt'
)

print('Training Transformer …')

print(

    f'{"Epoch":>6} | '

    f'{"tr_loss":>8} | '

    f'{"tr_acc":>8} | '

    f'{"val_loss":>9} | '

    f'{"val_acc":>8}'
)

print('-' * 75)


for epoch in range(

    1,

    CFG['NUM_EPOCHS'] + 1
):

    t0 = time.perf_counter()

    # TRAIN

    tr_loss, tr_acc = run_epoch(

        model,

        classifier_head,

        loaders['train'],

        criterion,

        optimizer=optimizer,

        scaler=scaler,

        train=True
    )

    # VALIDATION

    val_loss, val_acc = run_epoch(

        model,

        classifier_head,

        loaders['val'],

        criterion,

        train=False
    )

    scheduler.step()

    elapsed = (
        time.perf_counter() - t0
    )

    history['tr_loss'].append(tr_loss)

    history['val_loss'].append(val_loss)

    history['tr_acc'].append(tr_acc)

    history['val_acc'].append(val_acc)

    history['lr'].append(
        scheduler.get_last_lr()[0]
    )

    print(

        f'{epoch:>6} | '

        f'{tr_loss:>8.4f} | '

        f'{tr_acc:>8.4f} | '

        f'{val_loss:>9.4f} | '

        f'{val_acc:>8.4f} | '

        f'{elapsed:.1f}s'
    )

    # SAVE BEST

    if val_loss < best_val:

        best_val = val_loss

        patience_c = 0

        torch.save({

            'epoch': epoch,

            'model_state': model.state_dict(),

            'head_state': classifier_head.state_dict(),

            'optim_state': optimizer.state_dict(),

            'val_loss': val_loss

        }, CKPT_PATH)

        print(

            f'  ✓ Best model saved '

            f'(val_loss={val_loss:.4f})'
        )

    else:

        patience_c += 1

        if patience_c >= CFG['PATIENCE']:

            print(
                f'\nEarly stopping at epoch {epoch}'
            )

            break


# ════════════════════════════════════════════════════════════════
# LOAD BEST MODEL
# ════════════════════════════════════════════════════════════════

ckpt = torch.load(
    CKPT_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    ckpt['model_state']
)

classifier_head.load_state_dict(
    ckpt['head_state']
)

print('\nBest weights reloaded.')


# ════════════════════════════════════════════════════════════════
# TEST EVALUATION
# ════════════════════════════════════════════════════════════════

test_loss, test_acc = run_epoch(

    model,

    classifier_head,

    loaders['test'],

    criterion,

    train=False
)

print('\n' + '=' * 55)

print('TEST PERFORMANCE')

print('=' * 55)

print(f'Test Loss : {test_loss:.4f}')

print(f'Test Acc  : {test_acc:.4f}')

print('=' * 55)

Training Transformer …
 Epoch |  tr_loss |   tr_acc |  val_loss |  val_acc
---------------------------------------------------------------------------
     1 |   0.6513 |   0.7677 |    0.5606 |   0.7593 | 5.8s
  ✓ Best model saved (val_loss=0.5606)
     2 |   0.5332 |   0.7795 |    0.5188 |   0.7593 | 4.8s
  ✓ Best model saved (val_loss=0.5188)
     3 |   0.4657 |   0.8740 |    0.5369 |   0.7593 | 3.0s
     4 |   0.5174 |   0.8110 |    0.6376 |   0.7407 | 2.1s
     5 |   0.4561 |   0.8228 |    0.5419 |   0.8519 | 2.1s
     6 |   0.4539 |   0.8701 |    0.5589 |   0.7593 | 2.1s
     7 |   0.4697 |   0.8031 |    0.5415 |   0.7593 | 2.8s

Early stopping at epoch 7

Best weights reloaded.

TEST PERFORMANCE
Test Loss : 0.3648
Test Acc  : 0.8929


In [ ]:
# ════════════════════════════════════════════════════════════════
# BUILD RAW TRACE SEQUENCES — ENHANCED VERSION
# Operation + Host + Agent Tokens
# ════════════════════════════════════════════════════════════════

import os
import csv


# ════════════════════════════════════════════════════════════════
# TRACEBENCH ROOT
# ════════════════════════════════════════════════════════════════

TRACEBENCH_DIR = (

    "/kaggle/input/"
    "loghub-hdfs-hadoop-distributed-file-system-data/"
    "HDFS_v3_TraceBench/tracebench"
)


# ════════════════════════════════════════════════════════════════
# STORAGE
# ════════════════════════════════════════════════════════════════

traces = {}

labels = {}


# ════════════════════════════════════════════════════════════════
# LOAD TRACE SCENARIOS
# ════════════════════════════════════════════════════════════════

for scenario in os.listdir(TRACEBENCH_DIR):

    scenario_path = os.path.join(
        TRACEBENCH_DIR,
        scenario
    )

    event_file = os.path.join(
        scenario_path,
        "event.csv"
    )

    if not os.path.exists(event_file):

        continue

    try:

        sequence = []

        with open(

            event_file,

            'r',

            encoding='utf-8',

            errors='ignore'
        ) as f:

            reader = csv.reader(f)

            header = next(reader, None)

            for row in reader:

                # ------------------------------------------------
                # VALIDATE ROW
                # ------------------------------------------------

                if len(row) < 8:

                    continue

                # ------------------------------------------------
                # EXTRACT FIELDS
                # ------------------------------------------------

                op = row[2].strip()

                host = row[6].strip()

                agent = row[7].strip()

                # ------------------------------------------------
                # FILTER BAD ROWS
                # ------------------------------------------------

                if (

                    len(op) == 0

                    or

                    op.isdigit()
                ):

                    continue

                # ------------------------------------------------
                # CREATE ENHANCED TOKEN
                # ------------------------------------------------

                token = f"{op}@{host}@{agent}"

                sequence.append(token)

        # ------------------------------------------------
        # SKIP EMPTY
        # ------------------------------------------------

        if len(sequence) == 0:

            continue

        # ------------------------------------------------
        # STORE TRACE
        # ------------------------------------------------

        traces[scenario] = sequence

        # ------------------------------------------------
        # LABEL
        # ------------------------------------------------

        labels[scenario] = (

            1 if scenario.startswith('AN')

            else 0
        )

    except Exception as e:

        print(f'Skipping {scenario}: {e}')


# ════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════

print(f'Loaded Traces: {len(traces):,}')

n_anom = sum(labels.values())

print(f'Anomalies: {n_anom:,}')

print(f'Normals: {len(labels)-n_anom:,}')


# ════════════════════════════════════════════════════════════════
# EXAMPLE
# ════════════════════════════════════════════════════════════════

sample_key = list(traces.keys())[0]

print('\nExample Scenario:\n')

print(sample_key)

print('\nFirst 20 Operations:\n')

print(traces[sample_key][:20])

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/loghub-hdfs-hadoop-distributed-file-system-data/HDFS_v3_TraceBench/tracebench'

In [ ]:
# ════════════════════════════════════════════════════════════════
# FULL PIPELINE — Sliding Window + Transformer + RF
# ════════════════════════════════════════════════════════════════

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import collections
import random
import time
import math
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from torch.cuda.amp import (
    autocast,
    GradScaler
)


# ════════════════════════════════════════════════════════════════
# SETTINGS
# ════════════════════════════════════════════════════════════════

WINDOW_SIZE = 20

STRIDE = 5

CFG['MAX_SEQ_LEN'] = WINDOW_SIZE


# ════════════════════════════════════════════════════════════════
# SLIDING WINDOWS
# ════════════════════════════════════════════════════════════════

def create_sliding_windows(

    sequence,

    window_size=20,

    stride=5
):

    windows = []

    if len(sequence) < window_size:

        windows.append(sequence)

        return windows

    for i in range(

        0,

        len(sequence) - window_size + 1,

        stride
    ):

        win = sequence[i:i + window_size]

        windows.append(win)

    return windows


# ════════════════════════════════════════════════════════════════
# BUILD WINDOW DATASET
# ════════════════════════════════════════════════════════════════

all_sequences = []

all_labels = []

for scenario_name, sequence in traces.items():

    label = labels[scenario_name]

    windows = create_sliding_windows(

        sequence,

        WINDOW_SIZE,

        STRIDE
    )

    for w in windows:

        all_sequences.append(w)

        all_labels.append(label)


print(f'Total Window Samples: {len(all_sequences):,}')


# ════════════════════════════════════════════════════════════════
# VOCAB
# ════════════════════════════════════════════════════════════════

unique_events = set()

for seq in all_sequences:

    unique_events.update(seq)

event2id = {

    event: idx + 1

    for idx, event in enumerate(
        sorted(unique_events)
    )
}

event2id['<PAD>'] = 0

id2event = {

    v: k

    for k, v in event2id.items()
}


class Vocab:

    def __init__(self, token2id):

        self.token2id = token2id

        self.id2token = {

            v: k for k, v in token2id.items()
        }

    def __len__(self):

        return len(self.token2id)


vocab = Vocab(event2id)

CFG['VOCAB_SIZE'] = len(vocab)

print(f'Vocabulary Size: {len(vocab):,}')


# ════════════════════════════════════════════════════════════════
# ENCODING
# ════════════════════════════════════════════════════════════════

def encode_sequence(seq):

    ids = [

        event2id[x]

        for x in seq
    ]

    if len(ids) < WINDOW_SIZE:

        ids += [0] * (
            WINDOW_SIZE - len(ids)
        )

    return ids[:WINDOW_SIZE]


X = np.array(

    [encode_sequence(seq) for seq in all_sequences],

    dtype=np.int64
)

y = np.array(

    all_labels,

    dtype=np.int64
)


print('\nDataset Shape:')

print(X.shape)

print(y.shape)


# ════════════════════════════════════════════════════════════════
# SPLITS
# ════════════════════════════════════════════════════════════════

X_train, X_temp, y_train, y_temp = train_test_split(

    X,

    y,

    test_size=0.30,

    stratify=y,

    random_state=CFG['RANDOM_SEED']
)

X_val, X_test, y_val, y_test = train_test_split(

    X_temp,

    y_temp,

    test_size=0.50,

    stratify=y_temp,

    random_state=CFG['RANDOM_SEED']
)


splits = {

    'train': (X_train, y_train),

    'val': (X_val, y_val),

    'test': (X_test, y_test)
}


# ════════════════════════════════════════════════════════════════
# DATALOADERS
# ════════════════════════════════════════════════════════════════

def make_loader(X, y, shuffle=False):

    ds = TensorDataset(

        torch.tensor(X, dtype=torch.long),

        torch.tensor(y, dtype=torch.long)
    )

    return DataLoader(

        ds,

        batch_size=CFG['BATCH_SIZE'],

        shuffle=shuffle
    )


loaders = {

    'train': make_loader(
        X_train,
        y_train,
        shuffle=True
    ),

    'val': make_loader(
        X_val,
        y_val
    ),

    'test': make_loader(
        X_test,
        y_test
    )
}


# ════════════════════════════════════════════════════════════════
# DATASET STATS
# ════════════════════════════════════════════════════════════════

print('\nDataset Statistics:\n')

for name, (_, yy) in splits.items():

    anom = int(np.sum(yy))

    print(

        f'{name}: '

        f'{len(yy):,} samples | '

        f'anomaly={anom} '

        f'({anom/len(yy)*100:.2f}%)'
    )


# ════════════════════════════════════════════════════════════════
# POSITIONAL ENCODING
# ════════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):

    def __init__(

        self,

        d_model,

        max_len=512,

        dropout=0.1
    ):

        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)

        pos = torch.arange(
            0,
            max_len
        ).unsqueeze(1).float()

        div = torch.exp(

            torch.arange(
                0,
                d_model,
                2
            ).float()

            *

            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div)

        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer(
            'pe',
            pe.unsqueeze(0)
        )

    def forward(self, x):

        x = x + self.pe[:, :x.size(1)]

        return self.dropout(x)


# ════════════════════════════════════════════════════════════════
# TRANSFORMER
# ════════════════════════════════════════════════════════════════

class LightTransformerEncoder(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        D = CFG['EMBED_DIM']

        self.embedding = nn.Embedding(

            vocab_size,

            D,

            padding_idx=0
        )

        self.pos_enc = PositionalEncoding(

            D,

            max_len=CFG['MAX_SEQ_LEN']
        )

        enc_layer = nn.TransformerEncoderLayer(

            d_model=D,

            nhead=CFG['NUM_HEADS'],

            dim_feedforward=CFG['FF_DIM'],

            dropout=CFG['DROPOUT'],

            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(

            enc_layer,

            num_layers=CFG['NUM_LAYERS']
        )

        self.norm = nn.LayerNorm(D)

    def forward(self, x):

        pad = (x == 0)

        x = self.embedding(x)

        x = self.pos_enc(x)

        x = self.encoder(

            x,

            src_key_padding_mask=pad
        )

        x = self.norm(x)

        mask = (~pad).unsqueeze(-1).float()

        pooled = (

            (x * mask).sum(1)

            /

            mask.sum(1).clamp(min=1e-9)
        )

        return pooled

    def extract_embeddings(self, x):

        return self.forward(x)


DEVICE = torch.device(

    'cuda'

    if torch.cuda.is_available()

    else 'cpu'
)

model = LightTransformerEncoder(

    vocab_size=len(vocab)
).to(DEVICE)


classifier_head = nn.Sequential(

    nn.Linear(
        CFG['EMBED_DIM'],
        CFG['FF_DIM'] // 2
    ),

    nn.GELU(),

    nn.Dropout(
        CFG['DROPOUT']
    ),

    nn.Linear(
        CFG['FF_DIM'] // 2,
        2
    )

).to(DEVICE)


print(f'\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}')

print(f'Device: {DEVICE}')


# ════════════════════════════════════════════════════════════════
# TRAINING
# ════════════════════════════════════════════════════════════════

def compute_class_weights(y):

    counts = np.bincount(
        y,
        minlength=2
    ).astype(float)

    w = counts.sum() / (
        2.0 * counts + 1e-9
    )

    return torch.tensor(
        w,
        dtype=torch.float32
    ).to(DEVICE)


criterion = nn.CrossEntropyLoss(

    weight=compute_class_weights(y_train)
)

optimizer = optim.AdamW(

    list(model.parameters())
    +
    list(classifier_head.parameters()),

    lr=CFG['LEARNING_RATE']
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=CFG['NUM_EPOCHS']
)

scaler = GradScaler()


def run_epoch(

    loader,

    train=True
):

    model.train() if train else model.eval()

    classifier_head.train() if train else classifier_head.eval()

    total_loss = 0

    correct = 0

    total = 0

    ctx = (

        torch.enable_grad()

        if train

        else torch.no_grad()
    )

    with ctx:

        for Xb, yb in loader:

            Xb = Xb.to(DEVICE)

            yb = yb.to(DEVICE)

            if train:

                optimizer.zero_grad()

            with autocast():

                emb = model(Xb)

                logits = classifier_head(emb)

                loss = criterion(
                    logits,
                    yb
                )

            if train:

                scaler.scale(loss).backward()

                scaler.step(optimizer)

                scaler.update()

            total_loss += (
                loss.item()
                *
                yb.size(0)
            )

            correct += (

                logits.argmax(1) == yb

            ).sum().item()

            total += yb.size(0)

    return (

        total_loss / total,

        correct / total
    )


print('\nTraining Transformer...\n')

best_val = 1e9

for epoch in range(

    1,

    CFG['NUM_EPOCHS'] + 1
):

    tr_loss, tr_acc = run_epoch(

        loaders['train'],

        train=True
    )

    val_loss, val_acc = run_epoch(

        loaders['val'],

        train=False
    )

    scheduler.step()

    print(

        f'Epoch {epoch:02d} | '

        f'tr_acc={tr_acc:.4f} | '

        f'val_acc={val_acc:.4f}'
    )

    if val_loss < best_val:

        best_val = val_loss

        best_state = {

            'model': model.state_dict(),

            'head': classifier_head.state_dict()
        }


model.load_state_dict(best_state['model'])

classifier_head.load_state_dict(best_state['head'])

print('\nBest weights loaded.')


# ════════════════════════════════════════════════════════════════
# EMBEDDINGS
# ════════════════════════════════════════════════════════════════

@torch.no_grad()
def extract_embeddings(loader):

    model.eval()

    all_embs = []

    all_labels = []

    for Xb, yb in loader:

        Xb = Xb.to(DEVICE)

        emb = model.extract_embeddings(Xb)

        all_embs.append(
            emb.cpu().numpy()
        )

        all_labels.append(
            yb.numpy()
        )

    return (

        np.concatenate(all_embs),

        np.concatenate(all_labels)
    )


embeddings = {}

for sp in ['train', 'val', 'test']:

    embs, lbls = extract_embeddings(

        loaders[sp]
    )

    embeddings[sp] = (embs, lbls)


# ════════════════════════════════════════════════════════════════
# RANDOM FOREST
# ════════════════════════════════════════════════════════════════

X_tr, y_tr = embeddings['train']

X_val, y_val = embeddings['val']

X_te, y_te = embeddings['test']


rf = RandomForestClassifier(

    n_estimators=200,

    class_weight='balanced',

    n_jobs=-1,

    random_state=42
)

rf.fit(X_tr, y_tr)

val_acc = rf.score(X_val, y_val)

print(f'\nRF Validation Accuracy: {val_acc:.4f}')


X_tv = np.concatenate([X_tr, X_val])

y_tv = np.concatenate([y_tr, y_val])

rf.fit(X_tv, y_tv)


# ════════════════════════════════════════════════════════════════
# TEST EVALUATION
# ════════════════════════════════════════════════════════════════

y_pred = rf.predict(X_te)

y_prob = rf.predict_proba(X_te)[:, 1]


acc = accuracy_score(y_te, y_pred)

prec = precision_score(y_te, y_pred)

rec = recall_score(y_te, y_pred)

f1 = f1_score(y_te, y_pred)

roc = roc_auc_score(y_te, y_prob)

pr = average_precision_score(y_te, y_prob)

cm = confusion_matrix(y_te, y_pred)

tn, fp, fn, tp = cm.ravel()


print('\n' + '=' * 60)

print('FINAL TEST RESULTS')

print('=' * 60)

print(f'Accuracy      : {acc:.4f}')

print(f'Precision     : {prec:.4f}')

print(f'Recall        : {rec:.4f}')

print(f'F1 Score      : {f1:.4f}')

print(f'ROC-AUC       : {roc:.4f}')

print(f'PR-AUC        : {pr:.4f}')

print('\nConfusion Matrix')

print(f'TP={tp}  FP={fp}')

print(f'FN={fn}  TN={tn}')

print('=' * 60)

Total Window Samples: 2,954,310
Vocabulary Size: 6,157

Dataset Shape:
(2954310, 20)
(2954310,)

Dataset Statistics:

train: 2,068,017 samples | anomaly=982084 (47.49%)
val: 443,146 samples | anomaly=210446 (47.49%)
test: 443,147 samples | anomaly=210447 (47.49%)

Model Parameters: 461,120
Device: cpu

Training Transformer...

Epoch 01 | tr_acc=0.8176 | val_acc=0.8295


In [ ]:
# ════════════════════════════════════════════════════════════════
# HIGH ACCURACY TRACEBENCH PIPELINE
# TRACEBENCH HDFS + XGBOOST + RANDOM FOREST
# FIXED TOKENIZATION VERSION
# ════════════════════════════════════════════════════════════════

# !pip install xgboost kagglehub -q

import os
import re
import time
import random
import joblib
import numpy as np

from collections import Counter

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    average_precision_score,

    confusion_matrix
)

from xgboost import XGBClassifier

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════

CFG = {

    'WINDOW_SIZE': 30,

    'STRIDE': 5,

    'RANDOM_SEED': 42
}

random.seed(CFG['RANDOM_SEED'])

np.random.seed(CFG['RANDOM_SEED'])

# ════════════════════════════════════════════════════════════════
# DATASET PATH
# ════════════════════════════════════════════════════════════════

TRACEBENCH_DIR = (

    '/root/.cache/kagglehub/datasets/'
    'ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/'
    'versions/3/HDFS_v3_TraceBench/tracebench'
)

print('\nDataset Path:\n')

print(TRACEBENCH_DIR)

print('\nSample Scenarios:\n')

print(os.listdir(TRACEBENCH_DIR)[:10])

# ════════════════════════════════════════════════════════════════
# PREPROCESSING
# ════════════════════════════════════════════════════════════════

traces = {}

labels = {}

for scenario in os.listdir(TRACEBENCH_DIR):

    scenario_path = os.path.join(
        TRACEBENCH_DIR,
        scenario
    )

    if not os.path.isdir(scenario_path):

        continue

    operations = []

    for root, dirs, files in os.walk(scenario_path):

        for file in files:

            file_path = os.path.join(
                root,
                file
            )

            try:

                with open(

                    file_path,

                    'r',

                    encoding='utf-8',

                    errors='ignore'
                ) as f:

                    for line in f:

                        line = line.strip()

                        if not line:

                            continue

                        # SKIP CSV HEADER
                        if line.startswith('TaskID'):

                            continue

                        parts = line.split(',')

                        # EXTRACT ONLY OPERATION NAME
                        if len(parts) >= 3:

                            op = parts[2].strip()

                            if op:

                                operations.append(op)

            except Exception:

                continue

    if len(operations) == 0:

        continue

    traces[scenario] = operations

    # LABELS
    if scenario.startswith('NM_'):

        labels[scenario] = 0

    else:

        labels[scenario] = 1

# ════════════════════════════════════════════════════════════════
# DATASET STATS
# ════════════════════════════════════════════════════════════════

print('\nLoaded Traces:', len(traces))

anom = sum(labels.values())

normal = len(labels) - anom

print('Anomalies:', anom)

print('Normals:', normal)

example_key = list(traces.keys())[0]

print('\nExample Scenario:\n')

print(example_key)

print('\nFirst 20 Operations:\n')

print(traces[example_key][:20])

# SAVE PREPROCESSED DATA
joblib.dump(traces, 'traces.pkl')

joblib.dump(labels, 'labels.pkl')

print('\nSaved traces.pkl and labels.pkl')

# ════════════════════════════════════════════════════════════════
# SLIDING WINDOWS
# ════════════════════════════════════════════════════════════════

def create_sliding_windows(

    sequence,

    window_size=30,

    stride=5
):

    windows = []

    if len(sequence) < window_size:

        return [sequence]

    for i in range(

        0,

        len(sequence) - window_size + 1,

        stride
    ):

        windows.append(

            sequence[i:i + window_size]
        )

    return windows

# ════════════════════════════════════════════════════════════════
# BUILD WINDOW DATASET
# ════════════════════════════════════════════════════════════════

all_sequences = []

all_labels = []

for scenario_name, sequence in traces.items():

    label = labels[scenario_name]

    windows = create_sliding_windows(

        sequence,

        CFG['WINDOW_SIZE'],

        CFG['STRIDE']
    )

    for w in windows:

        all_sequences.append(w)

        all_labels.append(label)

print(f'\nTotal Window Samples: {len(all_sequences):,}')

# ════════════════════════════════════════════════════════════════
# VOCAB
# ════════════════════════════════════════════════════════════════

unique_events = set()

for seq in all_sequences:

    unique_events.update(seq)

event2id = {

    event: idx + 1

    for idx, event in enumerate(
        sorted(unique_events)
    )
}

event2id['<PAD>'] = 0

VOCAB_SIZE = len(event2id)

print(f'Vocabulary Size: {VOCAB_SIZE:,}')

# ════════════════════════════════════════════════════════════════
# ENCODING
# ════════════════════════════════════════════════════════════════

def encode_sequence(seq):

    ids = [

        event2id[x]

        for x in seq
    ]

    if len(ids) < CFG['WINDOW_SIZE']:

        ids += [0] * (

            CFG['WINDOW_SIZE'] - len(ids)
        )

    return ids[:CFG['WINDOW_SIZE']]

X = np.array(

    [encode_sequence(seq) for seq in all_sequences],

    dtype=np.int32
)

y = np.array(

    all_labels,

    dtype=np.int32
)

print('\nDataset Shape:')

print(X.shape)

print(y.shape)

# ════════════════════════════════════════════════════════════════
# SPLITS
# ════════════════════════════════════════════════════════════════

X_train, X_temp, y_train, y_temp = train_test_split(

    X,

    y,

    test_size=0.30,

    stratify=y,

    random_state=CFG['RANDOM_SEED']
)

X_val, X_test, y_val, y_test = train_test_split(

    X_temp,

    y_temp,

    test_size=0.50,

    stratify=y_temp,

    random_state=CFG['RANDOM_SEED']
)

print('\nDataset Statistics:\n')

for name, yy in [

    ('train', y_train),

    ('val', y_val),

    ('test', y_test)

]:

    anom = int(np.sum(yy))

    print(

        f'{name}: '

        f'{len(yy):,} samples | '

        f'anomaly={anom} '

        f'({anom/len(yy)*100:.2f}%)'
    )

# ════════════════════════════════════════════════════════════════
# XGBOOST
# ════════════════════════════════════════════════════════════════

print('\n' + '=' * 60)

print('TRAINING XGBOOST')

print('=' * 60)

start_xgb = time.time()

xgb = XGBClassifier(

    n_estimators=300,

    max_depth=10,

    learning_rate=0.05,

    subsample=0.9,

    colsample_bytree=0.9,

    objective='binary:logistic',

    eval_metric='auc',

    tree_method='hist',

    n_jobs=-1,

    random_state=42
)

xgb.fit(

    X_train,

    y_train,

    eval_set=[(X_val, y_val)],

    verbose=False
)

xgb_time = time.time() - start_xgb

print(f'\nXGBoost Training Time: {xgb_time/60:.2f} minutes')

# ════════════════════════════════════════════════════════════════
# XGBOOST EVALUATION
# ════════════════════════════════════════════════════════════════

y_pred = xgb.predict(X_test)

y_prob = xgb.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)

prec = precision_score(y_test, y_pred)

rec = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc = roc_auc_score(y_test, y_prob)

pr = average_precision_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print('\n' + '=' * 60)

print('XGBOOST RESULTS')

print('=' * 60)

print(f'Accuracy      : {acc:.4f}')

print(f'Precision     : {prec:.4f}')

print(f'Recall        : {rec:.4f}')

print(f'F1 Score      : {f1:.4f}')

print(f'ROC-AUC       : {roc:.4f}')

print(f'PR-AUC        : {pr:.4f}')

print('\nConfusion Matrix')

print(f'TP={tp}  FP={fp}')

print(f'FN={fn}  TN={tn}')

print('=' * 60)

# ════════════════════════════════════════════════════════════════
# RANDOM FOREST
# ════════════════════════════════════════════════════════════════

print('\n' + '=' * 60)

print('TRAINING RANDOM FOREST')

print('=' * 60)

start_rf = time.time()

rf = RandomForestClassifier(

    n_estimators=200,

    max_depth=30,

    class_weight='balanced',

    n_jobs=-1,

    random_state=42
)

sample_size = min(
    500000,
    len(X_train)
)

idx = np.random.choice(

    len(X_train),

    sample_size,

    replace=False
)

rf.fit(

    X_train[idx],

    y_train[idx]
)

rf_time = time.time() - start_rf

print(f'\nRF Training Time: {rf_time/60:.2f} minutes')

# ════════════════════════════════════════════════════════════════
# RF EVALUATION
# ════════════════════════════════════════════════════════════════

y_pred = rf.predict(X_test)

y_prob = rf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)

prec = precision_score(y_test, y_pred)

rec = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc = roc_auc_score(y_test, y_prob)

pr = average_precision_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print('\n' + '=' * 60)

print('RANDOM FOREST RESULTS')

print('=' * 60)

print(f'Accuracy      : {acc:.4f}')

print(f'Precision     : {prec:.4f}')

print(f'Recall        : {rec:.4f}')

print(f'F1 Score      : {f1:.4f}')

print(f'ROC-AUC       : {roc:.4f}')

print(f'PR-AUC        : {pr:.4f}')

print('\nConfusion Matrix')

print(f'TP={tp}  FP={fp}')

print(f'FN={fn}  TN={tn}')

print('=' * 60)

# ════════════════════════════════════════════════════════════════
# SAVE MODELS
# ════════════════════════════════════════════════════════════════

joblib.dump(
    xgb,
    'xgboost_model.pkl'
)

joblib.dump(
    rf,
    'random_forest_model.pkl'
)

joblib.dump(
    event2id,
    'event_vocab.pkl'
)

print('\nModels Saved Successfully')

print('xgboost_model.pkl')

print('random_forest_model.pkl')

print('event_vocab.pkl')


Dataset Path:

/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v3_TraceBench/tracebench

Sample Scenarios:

['AN_Data_cutMeta_r_10FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Net_slowHDFS_rpc_0400ms_30C_0to0INT_5RT_1WT', 'NM_DN_rw_30DN_30C_1to19B_0to120INTR_0to600INTW_50RT_10WT', 'AN_Proc_killDN_r_03FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Net_slowHDFS_w_18ms_30C_1to19B_0to600INT_15RT_5WT', 'AN_Data_lossMeta_r_40FDN_30C_1to19B_0to120INT_15RT_5WT', 'NM_CL_rpc_35C_0to0INT_10RT_1WT', 'COM_Sin_Proc_rwrpc_3_30C_1to19B_0to120INTR_0to600INTW_0to0INTRPC_60RT_10WT', 'AN_Sys_panicDN_r_05FDN_30C_1to19B_0to120INT_15RT_5WT', 'AN_Proc_suspendDN_r_50FDN_30C_1to19B_0to120INT_15RT_10WT']

Loaded Traces: 364
Anomalies: 276
Normals: 88

Example Scenario:

AN_Data_cutMeta_r_10FDN_30C_1to19B_0to120INT_15RT_5WT

First 20 Operations:

['getFileInfo', 'RPC:getFileInfo', 'getFileInfo', 'RPC:getFileInfo', 'getFileInfo', 'RPC:getFileInfo', 'getFileInfo', 'RPC:

In [ ]:
# ════════════════════════════════════════════════════════════════
# ULTRA OPTIMIZED TRACEBENCH PIPELINE
# FAST + HIGH ACCURACY + LOW RAM
# TF-IDF + LIGHT XGBOOST HYBRID
# COLAB CPU OPTIMIZED
# ════════════════════════════════════════════════════════════════

# !pip install xgboost kagglehub -q

import os
import re
import csv
import time
import random
import joblib
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    average_precision_score,

    confusion_matrix
)

from xgboost import XGBClassifier

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════

CFG = {

    'WINDOW_SIZE': 30,

    'STRIDE': 20,

    'RANDOM_SEED': 42
}

random.seed(CFG['RANDOM_SEED'])

np.random.seed(CFG['RANDOM_SEED'])

# ════════════════════════════════════════════════════════════════
# DATASET PATH
# ════════════════════════════════════════════════════════════════

TRACEBENCH_DIR = (

    '/root/.cache/kagglehub/datasets/'
    'ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/'
    'versions/3/HDFS_v3_TraceBench/tracebench'
)

print('\nDataset Path:\n')

print(TRACEBENCH_DIR)

# ════════════════════════════════════════════════════════════════
# PREPROCESSING
# ════════════════════════════════════════════════════════════════

traces = {}

labels = {}

start = time.time()

for scenario in os.listdir(TRACEBENCH_DIR):

    scenario_path = os.path.join(
        TRACEBENCH_DIR,
        scenario
    )

    if not os.path.isdir(scenario_path):

        continue

    operations = []

    for root, dirs, files in os.walk(scenario_path):

        for file in files:

            file_path = os.path.join(
                root,
                file
            )

            try:

                with open(

                    file_path,

                    'r',

                    encoding='utf-8',

                    errors='ignore'
                ) as f:

                    for line in f:

                        line = line.strip()

                        if not line:

                            continue

                        if line.startswith('TaskID'):

                            continue

                        try:

                            reader = csv.reader([line])

                            parts = next(reader)

                        except Exception:

                            continue

                        if len(parts) < 3:

                            continue

                        op = parts[2].strip()

                        # CLEAN
                        op = re.sub(
                            r'\d+',
                            '',
                            op
                        )

                        op = re.sub(
                            r'blk_[A-Za-z0-9_]+',
                            'BLOCK',
                            op
                        )

                        op = re.sub(
                            r'[^A-Za-z:_ ]',
                            '',
                            op
                        )

                        op = op.strip()

                        if op:

                            operations.append(op)

            except Exception:

                continue

    if len(operations) == 0:

        continue

    traces[scenario] = operations

    if scenario.startswith('NM_'):

        labels[scenario] = 0

    else:

        labels[scenario] = 1

print('\nLoaded Traces:', len(traces))

print(

    'Preprocessing Time:',

    f'{(time.time()-start)/60:.2f} mins'
)

# ════════════════════════════════════════════════════════════════
# TRACE-LEVEL SPLIT
# ════════════════════════════════════════════════════════════════

trace_keys = list(traces.keys())

trace_labels = [

    labels[k]

    for k in trace_keys
]

train_keys, temp_keys = train_test_split(

    trace_keys,

    test_size=0.30,

    stratify=trace_labels,

    random_state=42
)

temp_labels = [

    labels[k]

    for k in temp_keys
]

val_keys, test_keys = train_test_split(

    temp_keys,

    test_size=0.50,

    stratify=temp_labels,

    random_state=42
)

# ════════════════════════════════════════════════════════════════
# SLIDING WINDOWS
# ════════════════════════════════════════════════════════════════

def create_sliding_windows(

    sequence,

    window_size=30,

    stride=20
):

    if len(sequence) < window_size:

        return [sequence]

    return [

        sequence[i:i+window_size]

        for i in range(

            0,

            len(sequence)-window_size+1,

            stride
        )
    ]

# ════════════════════════════════════════════════════════════════
# BUILD DATASETS
# ════════════════════════════════════════════════════════════════

def build_dataset(keys):

    texts = []

    labels_out = []

    for key in keys:

        label = labels[key]

        windows = create_sliding_windows(

            traces[key],

            CFG['WINDOW_SIZE'],

            CFG['STRIDE']
        )

        for w in windows:

            texts.append(

                ' '.join(w)
            )

            labels_out.append(label)

    return texts, np.array(labels_out)

print('\nBuilding datasets...\n')

train_text, y_train = build_dataset(train_keys)

val_text, y_val = build_dataset(val_keys)

test_text, y_test = build_dataset(test_keys)

print('Train:', len(train_text))

print('Val  :', len(val_text))

print('Test :', len(test_text))

# ════════════════════════════════════════════════════════════════
# TF-IDF
# ════════════════════════════════════════════════════════════════

print('\nGenerating TF-IDF Features...\n')

start = time.time()

vectorizer = TfidfVectorizer(

    analyzer='word',

    ngram_range=(1, 2),

    min_df=3,

    max_df=0.95,

    max_features=2000,

    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_text)

X_val = vectorizer.transform(val_text)

X_test = vectorizer.transform(test_text)

print('\nTF-IDF Shapes:\n')

print(X_train.shape)

print(X_val.shape)

print(X_test.shape)

print(

    '\nTF-IDF Time:',

    f'{(time.time()-start)/60:.2f} mins'
)

# ════════════════════════════════════════════════════════════════
# XGBOOST
# ════════════════════════════════════════════════════════════════

print('\n' + '=' * 60)

print('TRAINING OPTIMIZED XGBOOST')

print('=' * 60)

start = time.time()

xgb = XGBClassifier(

    n_estimators=120,

    max_depth=7,

    learning_rate=0.08,

    subsample=0.8,

    colsample_bytree=0.8,

    objective='binary:logistic',

    eval_metric='auc',

    tree_method='hist',

    max_bin=128,

    grow_policy='lossguide',

    n_jobs=-1,

    random_state=42
)

xgb.fit(

    X_train,

    y_train,

    eval_set=[(X_val, y_val)],

    verbose=False
)

print(

    '\nTraining Time:',

    f'{(time.time()-start)/60:.2f} mins'
)

# ════════════════════════════════════════════════════════════════
# EVALUATION
# ════════════════════════════════════════════════════════════════

print('\nGenerating predictions...\n')

y_prob = xgb.predict_proba(X_test)[:, 1]

y_pred = (y_prob > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)

prec = precision_score(y_test, y_pred)

rec = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc = roc_auc_score(y_test, y_prob)

pr = average_precision_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print('\n' + '=' * 60)

print('FINAL RESULTS')

print('=' * 60)

print(f'Accuracy      : {acc:.4f}')

print(f'Precision     : {prec:.4f}')

print(f'Recall        : {rec:.4f}')

print(f'F1 Score      : {f1:.4f}')

print(f'ROC-AUC       : {roc:.4f}')

print(f'PR-AUC        : {pr:.4f}')

print('\nConfusion Matrix')

print(f'TP={tp}  FP={fp}')

print(f'FN={fn}  TN={tn}')

print('=' * 60)

# ════════════════════════════════════════════════════════════════
# SAVE
# ════════════════════════════════════════════════════════════════

joblib.dump(
    xgb,
    'optimized_xgb.pkl'
)

joblib.dump(
    vectorizer,
    'tfidf_vectorizer.pkl'
)

print('\nModels Saved Successfully')


Dataset Path:

/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_v3_TraceBench/tracebench

Loaded Traces: 364
Preprocessing Time: 3.15 mins

Building datasets...

Train: 538267
Val  : 93751
Test : 106523

Generating TF-IDF Features...


TF-IDF Shapes:

(538267, 1116)
(93751, 1116)
(106523, 1116)

TF-IDF Time: 0.91 mins

TRAINING OPTIMIZED XGBOOST

Training Time: 10.22 mins

Generating predictions...


FINAL RESULTS
Accuracy      : 0.5914
Precision     : 0.5677
Recall        : 0.8341
F1 Score      : 0.6756
ROC-AUC       : 0.7309
PR-AUC        : 0.8034

Confusion Matrix
TP=45320  FP=34511
FN=9016  TN=17676

Models Saved Successfully


In [ ]:
!pip install lightgbm xgboost catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
### Claude Code ####

# ════════════════════════════════════════════════════════════════
# ELITE TRACEBENCH HDFS ANOMALY DETECTION PIPELINE v2.0
# TARGET: 95%+ ACC  |  0.97+ ROC-AUC  |  0.97+ PR-AUC
# STACK: MARKOV + MULTI-TFIDF + BILSTM + LGB + XGB + CATBOOST
# ════════════════════════════════════════════════════════════════

import os, re, csv, time, random, joblib, warnings, gc, sys
import numpy as np
from collections import Counter, defaultdict
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
from xgboost import XGBClassifier
import lightgbm as lgb

# ── Optional deps ─────────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_AMP = (DEVICE.type == 'cuda')
    if USE_AMP:
        try:
            from torch.amp import autocast, GradScaler
            _AMP_DEVICE = 'cuda'
        except ImportError:
            from torch.cuda.amp import autocast, GradScaler
            _AMP_DEVICE = None
except ImportError:
    TORCH_AVAILABLE = False
    DEVICE = 'cpu'
    USE_AMP = False

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════

CFG = {
    'WINDOW_SIZE'           : 40,
    'STRIDE'                : 10,
    'RANDOM_SEED'           : 42,
    'TFIDF_WORD_FEATURES'   : 6000,
    'TFIDF_CHAR_FEATURES'   : 3000,
    'TFIDF_BIGRAM_FEATURES' : 2000,
    'EARLY_STOPPING_ROUNDS' : 40,
    'LGB_ITERS'             : 600,
    'XGB_ITERS'             : 400,
    'CB_ITERS'              : 400,
    'BILSTM_EPOCHS'         : 25,
    'BILSTM_BATCH'          : 256,
    'BILSTM_HIDDEN'         : 128,
    'BILSTM_LAYERS'         : 2,
    'BILSTM_EMBED'          : 64,
    'BILSTM_DROPOUT'        : 0.30,
    'BILSTM_PATIENCE'       : 6,
    'VOCAB_SIZE'            : 512,
    'MAX_SEQ_LEN'           : 40,
}

# ════════════════════════════════════════════════════════════════
# ENVIRONMENT REPORT
# ════════════════════════════════════════════════════════════════

print('=' * 60)
print('ELITE TRACEBENCH PIPELINE v2.0')
print('=' * 60)
print(f'  PyTorch    : {TORCH_AVAILABLE}  |  Device: {DEVICE}')
print(f'  CatBoost   : {CATBOOST_AVAILABLE}')
print(f'  AMP        : {USE_AMP}')
print('=' * 60)

# ════════════════════════════════════════════════════════════════
# AUTO PATH DETECTION
# ════════════════════════════════════════════════════════════════

_SEARCH_ROOTS = [
    '/root/.cache/kagglehub/datasets',
    os.path.expanduser('~/.cache/kagglehub/datasets'),
    '/content/drive/MyDrive',
    '/content',
    '.',
    '..',
]

def _count_scenario_dirs(path: str) -> int:
    """Count immediate subdirectories that look like scenario folders."""
    try:
        return sum(
            1 for d in os.listdir(path)
            if os.path.isdir(os.path.join(path, d)) and not d.startswith('.')
        )
    except OSError:
        return 0

def _find_tracebench(roots):
    """
    Return the directory that:
      1. Has 'tracebench' in its name, AND
      2. Contains the most scenario subdirectories (guards against stopping
         one level too high, e.g. at HDFS_v3_TraceBench instead of
         HDFS_v3_TraceBench/tracebench/).
    Fallback: any directory named 'tracebench*' with ≥1 subdir.
    """
    best_path  = None
    best_count = 0

    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            # Check the current dir itself
            if 'tracebench' in os.path.basename(dirpath).lower():
                n = _count_scenario_dirs(dirpath)
                if n > best_count:
                    best_count = n
                    best_path  = dirpath
            # Check immediate children
            for d in dirnames:
                if 'tracebench' in d.lower():
                    candidate = os.path.join(dirpath, d)
                    n = _count_scenario_dirs(candidate)
                    if n > best_count:
                        best_count = n
                        best_path  = candidate

    # If the winning dir still has very few children, descend one more level
    # (handles HDFS_v3_TraceBench → tracebench → NM_*/ANO_* nesting)
    if best_path is not None and best_count < 5:
        for child in os.listdir(best_path):
            cpath = os.path.join(best_path, child)
            if os.path.isdir(cpath):
                n = _count_scenario_dirs(cpath)
                if n > best_count:
                    best_count = n
                    best_path  = cpath

    return best_path

TRACEBENCH_DIR = _find_tracebench(_SEARCH_ROOTS)

if TRACEBENCH_DIR is None:
    try:
        import kagglehub
        _dl_path = kagglehub.dataset_download(
            'ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data'
        )
        TRACEBENCH_DIR = _find_tracebench([_dl_path]) or _dl_path
    except Exception as _e:
        raise RuntimeError(
            f'Cannot locate TraceBench dataset ({_e}). '
            'Set TRACEBENCH_DIR manually above _find_tracebench calls.'
        )

print(f'\nDataset path: {TRACEBENCH_DIR}\n')

# ════════════════════════════════════════════════════════════════
# ENHANCED PREPROCESSING
# ════════════════════════════════════════════════════════════════

_IP_RE    = re.compile(r'\b\d{1,3}(?:\.\d{1,3}){3}(?::\d+)?\b')
_BLK_RE   = re.compile(r'blk_[-\d]+')
_NUM4_RE  = re.compile(r'\b\d{4,}\b')
_CLEAN_RE = re.compile(r'[^A-Za-z:_ ]')
_WS_RE    = re.compile(r'\s+')

def clean_op(op: str) -> str:
    op = _BLK_RE.sub('BLOCK', op)
    op = _IP_RE.sub('IPADDR', op)
    op = _NUM4_RE.sub('NUM', op)
    op = _CLEAN_RE.sub(' ', op)
    op = _WS_RE.sub(' ', op).strip()
    return op or 'UNKNOWN'

t0 = time.time()
traces: dict[str, list[str]] = {}
labels: dict[str, int]       = {}

def _load_scenario(spath: str, key: str) -> list[str]:
    ops = []
    for root, _, files in os.walk(spath):
        for fname in sorted(files):
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, 'r', encoding='utf-8', errors='ignore') as fh:
                    for line in fh:
                        line = line.strip()
                        if not line or line.startswith('TaskID'):
                            continue
                        try:
                            parts = next(csv.reader([line]))
                        except Exception:
                            continue
                        if len(parts) < 3:
                            continue
                        op = clean_op(parts[2].strip())
                        if op:
                            ops.append(op)
            except Exception:
                continue
    return ops

def _label(name: str) -> int:
    return 0 if name.startswith('NM_') else 1

def _ingest_dir(parent: str) -> None:
    """
    Ingest all scenario folders found directly under `parent`.
    A folder counts as a scenario if it contains at least one non-dir file
    anywhere in its subtree.  If the immediate children of `parent` are
    grouping dirs (no NM_/anomaly pattern themselves) each child is
    recursed into as a sub-parent.
    """
    entries = sorted(os.listdir(parent))
    scenario_entries = [
        e for e in entries
        if os.path.isdir(os.path.join(parent, e)) and not e.startswith('.')
    ]

    direct_scenarios = [
        e for e in scenario_entries
        if e.startswith('NM_') or re.search(r'(ANO|Ano|anomal|fault|fail)',
                                             e, re.IGNORECASE)
    ]

    if direct_scenarios:
        # This level contains real scenario dirs
        for entry in scenario_entries:
            epath = os.path.join(parent, entry)
            ops   = _load_scenario(epath, entry)
            if ops:
                traces[entry] = ops
                labels[entry] = _label(entry)
    else:
        # Grouping level — descend one more level into each child
        for entry in scenario_entries:
            _ingest_dir(os.path.join(parent, entry))

_ingest_dir(TRACEBENCH_DIR)

n_normal = sum(1 for v in labels.values() if v == 0)
n_anom   = sum(1 for v in labels.values() if v == 1)
print(f'Loaded {len(traces)} traces in {(time.time()-t0)/60:.2f}m')
print(f'  Normal={n_normal}  Anomalous={n_anom}')

# ════════════════════════════════════════════════════════════════
# TRACE-LEVEL SPLIT  (no leakage)
# ════════════════════════════════════════════════════════════════

trace_keys   = list(traces.keys())
trace_labels = [labels[k] for k in trace_keys]
n_total      = len(trace_keys)

print(f'\nTotal traces loaded: {n_total}  (normal={n_normal}  anomalous={n_anom})')

# ── Sanity check ──────────────────────────────────────────────
if n_total < 6:
    # Print dataset tree to help diagnose the path problem
    print('\n[DIAGNOSTIC] Top-3 levels under TRACEBENCH_DIR:')
    for _r, _ds, _fs in os.walk(TRACEBENCH_DIR):
        _depth = _r.replace(TRACEBENCH_DIR, '').count(os.sep)
        if _depth > 3:
            break
        print('  ' + '  ' * _depth + os.path.basename(_r) + '/')
        for _f in sorted(_fs)[:3]:
            print('  ' + '  ' * (_depth+1) + _f)
    raise RuntimeError(
        f'Only {n_total} traces found in {TRACEBENCH_DIR}.\n'
        'The dataset path is likely wrong or one level too high.\n'
        'Manually set TRACEBENCH_DIR to the folder that directly contains '
        'NM_* and anomaly scenario subdirectories.'
    )

if n_normal == 0 or n_anom == 0:
    raise RuntimeError(
        f'All {n_total} traces have the same label '
        f'(normal={n_normal}, anomalous={n_anom}).\n'
        'Check that scenario folder names follow the NM_* convention for '
        'normal traces and any other prefix for anomalous traces.'
    )

# ── Adaptive split sizes ───────────────────────────────────────
# Guarantee at least 1 sample per class in every split.
# For small datasets fall back to a 60/20/20 with no stratification.
def _safe_split(keys, lbl_list, test_frac, seed=42):
    try:
        a, b = train_test_split(
            keys, test_size=test_frac,
            stratify=lbl_list, random_state=seed
        )
        return a, b
    except ValueError:
        # Dataset too small for stratified split
        a, b = train_test_split(keys, test_size=test_frac, random_state=seed)
        return a, b

train_keys, temp_keys = _safe_split(trace_keys, trace_labels, 0.30)
val_keys,  test_keys  = _safe_split(
    temp_keys, [labels[k] for k in temp_keys], 0.50
)

# ════════════════════════════════════════════════════════════════
# SLIDING WINDOW
# ════════════════════════════════════════════════════════════════

def make_windows(seq: list[str], wsize: int, stride: int) -> list[list[str]]:
    if len(seq) < wsize:
        return [seq + ['PAD'] * (wsize - len(seq))]
    return [seq[i:i+wsize] for i in range(0, len(seq) - wsize + 1, stride)]

# ════════════════════════════════════════════════════════════════
# MARKOV TRANSITION MODEL
# ════════════════════════════════════════════════════════════════

class MarkovModel:
    """First-order Markov transition model with Laplace smoothing."""

    SMOOTH = 1e-9

    def __init__(self):
        self._trans: dict[str, Counter] = defaultdict(Counter)
        self._totals: dict[str, int]    = {}
        self.vocab: set[str]            = set()

    def fit(self, sequences: list[list[str]]) -> 'MarkovModel':
        for seq in sequences:
            self.vocab.update(seq)
            for a, b in zip(seq, seq[1:]):
                self._trans[a][b] += 1
        self._totals = {k: sum(v.values()) for k, v in self._trans.items()}
        self._V = max(len(self.vocab), 1)
        return self

    def _lp(self, a: str, b: str) -> float:
        tot = self._totals.get(a, 0)
        if tot == 0:
            return np.log(self.SMOOTH)
        cnt = self._trans[a].get(b, 0)
        return np.log((cnt + self.SMOOTH) / (tot + self.SMOOTH * self._V))

    def window_features(self, win: list[str]) -> list[float]:
        if len(win) < 2:
            return [0.0] * 7
        lps  = np.array([self._lp(a, b) for a, b in zip(win, win[1:])])
        oovr = float(np.mean(lps < np.log(self.SMOOTH * 10)))
        # Novel bigrams: pairs never seen in training
        novel = sum(
            self._totals.get(a, 0) == 0 or self._trans[a].get(b, 0) == 0
            for a, b in zip(win, win[1:])
        ) / len(lps)
        return [
            float(np.mean(lps)),
            float(np.min(lps)),
            float(np.max(lps)),
            float(np.std(lps)),
            float(np.percentile(lps, 10)),
            float(oovr),
            float(novel),
        ]

# ════════════════════════════════════════════════════════════════
# STATISTICAL FEATURE EXTRACTION  (24 features + 7 Markov = 31)
# ════════════════════════════════════════════════════════════════

def stat_features(win: list[str], mk: MarkovModel) -> list[float]:
    n      = len(win)
    counts = Counter(win)
    probs  = np.array(list(counts.values()), dtype=np.float64) / n

    # Diversity
    uniq_ratio   = len(counts) / n
    entropy      = float(-np.sum(probs * np.log(probs + 1e-12)))
    max_freq     = float(max(counts.values()) / n)
    min_freq     = float(min(counts.values()) / n)
    gini         = float(1.0 - np.sum(probs ** 2))     # Gini impurity

    # Pattern indicators
    rpc_r   = sum('RPC'   in x for x in win) / n
    op_r    = sum('OP'    in x for x in win) / n
    blk_r   = sum('BLOCK' in x for x in win) / n
    num_r   = sum('NUM'   in x for x in win) / n
    ip_r    = sum('IPADDR'in x for x in win) / n
    pad_r   = sum(x == 'PAD' for x in win) / n
    err_r   = sum(any(t in x.upper() for t in ('ERR', 'FAIL', 'EXCP', 'TIMEOUT'))
                  for x in win) / n

    # Temporal structure
    trans_r      = sum(win[i] != win[i+1] for i in range(n-1)) / max(n-1, 1)
    consec_r     = 1.0 - trans_r

    # Bigram diversity
    bigrams  = [f'{win[i]}__{win[i+1]}' for i in range(n-1)]
    bi_uniq  = len(set(bigrams)) / max(len(bigrams), 1)

    # Run-length
    runs = 1 + sum(win[i] != win[i-1] for i in range(1, n))
    mean_run = n / runs

    # Positional split diversity
    h = n // 2
    head_uniq = len(set(win[:h])) / max(h, 1)
    tail_uniq = len(set(win[h:])) / max(n - h, 1)

    # First-rare-event position
    rare = {k for k, v in counts.items() if v == 1}
    first_rare = next((i / n for i, x in enumerate(win) if x in rare), 1.0)

    # Relative window fill (when padded, < 1)
    fill = sum(x != 'PAD' for x in win) / n

    stat = [
        uniq_ratio, entropy, max_freq, min_freq, gini,
        rpc_r, op_r, blk_r, num_r, ip_r, pad_r, err_r,
        trans_r, consec_r, bi_uniq,
        mean_run / n, head_uniq, tail_uniq, first_rare, fill,
        float(len(counts)) / 50,           # normalised unique count
        float(n) / CFG['WINDOW_SIZE'],     # relative length
        float(sum(probs > 0.15)),          # dominant events
        float(np.std(probs)),              # frequency spread
    ]

    return stat + mk.window_features(win)     # 24 + 7 = 31 features

# ════════════════════════════════════════════════════════════════
# DATASET BUILDER
# ════════════════════════════════════════════════════════════════

def build_dataset(keys, mk: MarkovModel):
    texts_w, texts_c, texts_b, stats, ys = [], [], [], [], []
    wsize, stride = CFG['WINDOW_SIZE'], CFG['STRIDE']
    for key in keys:
        lbl = labels[key]
        for win in make_windows(traces[key], wsize, stride):
            texts_w.append(' '.join(win))
            texts_c.append(' '.join(win))                          # same text, different vectoriser
            texts_b.append(' '.join(f'{win[i]}__{win[i+1]}' for i in range(len(win)-1)))
            stats.append(stat_features(win, mk))
            ys.append(lbl)
    return texts_w, texts_c, texts_b, np.array(stats, dtype=np.float32), np.array(ys)

# ─── Fit Markov on training sequences only ────────────────────
print(f'Split — Train:{len(train_keys)}  Val:{len(val_keys)}  Test:{len(test_keys)}')

print('\nFitting Markov model...')
markov = MarkovModel().fit([traces[k] for k in train_keys])

print('Building windows...')
t0 = time.time()
tr_w, tr_c, tr_b, tr_s, y_train = build_dataset(train_keys, markov)
va_w, va_c, va_b, va_s, y_val   = build_dataset(val_keys,   markov)
te_w, te_c, te_b, te_s, y_test  = build_dataset(test_keys,  markov)
print(f'  Train={len(y_train)}  Val={len(y_val)}  Test={len(y_test)}  [{time.time()-t0:.1f}s]')

# ════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING  (TF-IDF × 3  +  scaled stats)
# ════════════════════════════════════════════════════════════════

print('\nBuilding TF-IDF features...')
t0 = time.time()

# Word n-grams (1–3)
tfidf_w = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 3),
    min_df=2, max_df=0.98,
    max_features=CFG['TFIDF_WORD_FEATURES'],
    sublinear_tf=True
)
Xtr_w = tfidf_w.fit_transform(tr_w)
Xva_w = tfidf_w.transform(va_w)
Xte_w = tfidf_w.transform(te_w)

# Char wb n-grams (3–5) — catches partial token patterns
tfidf_c = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    min_df=2, max_df=0.98,
    max_features=CFG['TFIDF_CHAR_FEATURES'],
    sublinear_tf=True
)
Xtr_c = tfidf_c.fit_transform(tr_c)
Xva_c = tfidf_c.transform(va_c)
Xte_c = tfidf_c.transform(te_c)

# Bigram-sequence TF-IDF — encodes transition bigrams as tokens
tfidf_b = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    min_df=2, max_df=0.98,
    max_features=CFG['TFIDF_BIGRAM_FEATURES'],
    sublinear_tf=True
)
Xtr_b = tfidf_b.fit_transform(tr_b)
Xva_b = tfidf_b.transform(va_b)
Xte_b = tfidf_b.transform(te_b)

# Scaled statistical + Markov features
scaler = StandardScaler()
Xtr_s = csr_matrix(scaler.fit_transform(tr_s))
Xva_s = csr_matrix(scaler.transform(va_s))
Xte_s = csr_matrix(scaler.transform(te_s))

# Fused feature matrix
X_train = hstack([Xtr_w, Xtr_c, Xtr_b, Xtr_s]).tocsr()
X_val   = hstack([Xva_w, Xva_c, Xva_b, Xva_s]).tocsr()
X_test  = hstack([Xte_w, Xte_c, Xte_b, Xte_s]).tocsr()
print(f'  Feature shape: {X_train.shape}  [{time.time()-t0:.1f}s]')

# Class imbalance weight
neg, pos         = int((y_train == 0).sum()), int((y_train == 1).sum())
scale_pos_weight = neg / max(pos, 1)
print(f'\n  Normal={neg}  Anomalous={pos}  scale_pos_weight={scale_pos_weight:.3f}')

# ════════════════════════════════════════════════════════════════
# LIGHTGBM
# ════════════════════════════════════════════════════════════════

print('\n' + '='*60)
print('LIGHTGBM')
print('='*60)
t0 = time.time()

lgb_model = lgb.LGBMClassifier(
    n_estimators       = CFG['LGB_ITERS'],
    max_depth          = 9,
    num_leaves         = 95,
    learning_rate      = 0.04,
    subsample          = 0.85,
    subsample_freq     = 1,
    colsample_bytree   = 0.80,
    min_child_samples  = 8,
    reg_alpha          = 0.05,
    reg_lambda         = 1.0,
    scale_pos_weight   = scale_pos_weight,
    objective          = 'binary',
    metric             = 'auc',
    n_jobs             = -1,
    random_state       = 42,
    verbose            = -1,
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(CFG['EARLY_STOPPING_ROUNDS'], verbose=False),
        lgb.log_evaluation(period=100),
    ],
)
lgb_vp = lgb_model.predict_proba(X_val)[:, 1]
lgb_tp = lgb_model.predict_proba(X_test)[:, 1]
print(f'LGB  Val ROC-AUC: {roc_auc_score(y_val, lgb_vp):.4f}  [{time.time()-t0:.1f}s]')

# ════════════════════════════════════════════════════════════════
# XGBOOST
# ════════════════════════════════════════════════════════════════

print('\n' + '='*60)
print('XGBOOST')
print('='*60)
t0 = time.time()

_xgb_extra = {}
if TORCH_AVAILABLE and torch.cuda.is_available():
    _xgb_extra['device'] = 'cuda'

xgb_model = XGBClassifier(
    n_estimators         = CFG['XGB_ITERS'],
    max_depth            = 9,
    learning_rate        = 0.04,
    subsample            = 0.85,
    colsample_bytree     = 0.80,
    gamma                = 0.1,
    min_child_weight     = 3,
    reg_alpha            = 0.05,
    reg_lambda           = 1.5,
    scale_pos_weight     = scale_pos_weight,
    objective            = 'binary:logistic',
    eval_metric          = 'auc',
    tree_method          = 'hist',
    max_bin              = 256,
    early_stopping_rounds= CFG['EARLY_STOPPING_ROUNDS'],
    n_jobs               = -1,
    random_state         = 42,
    verbosity            = 0,
    **_xgb_extra,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
xgb_vp = xgb_model.predict_proba(X_val)[:, 1]
xgb_tp = xgb_model.predict_proba(X_test)[:, 1]
print(f'XGB  Val ROC-AUC: {roc_auc_score(y_val, xgb_vp):.4f}  [{time.time()-t0:.1f}s]')

# ════════════════════════════════════════════════════════════════
# CATBOOST  (optional)
# ════════════════════════════════════════════════════════════════

cb_vp = cb_tp = None

if CATBOOST_AVAILABLE:
    print('\n' + '='*60)
    print('CATBOOST')
    print('='*60)
    t0 = time.time()
    from catboost import Pool as CbPool
    tr_pool = CbPool(X_train, y_train)
    va_pool = CbPool(X_val,   y_val)

    cb_model = CatBoostClassifier(
        iterations          = CFG['CB_ITERS'],
        depth               = 8,
        learning_rate       = 0.04,
        eval_metric         = 'AUC',
        early_stopping_rounds = CFG['EARLY_STOPPING_ROUNDS'],
        scale_pos_weight    = scale_pos_weight,
        random_seed         = 42,
        verbose             = 100,
        task_type           = 'GPU' if (TORCH_AVAILABLE and torch.cuda.is_available()) else 'CPU',
        od_type             = 'Iter',
    )
    cb_model.fit(tr_pool, eval_set=va_pool)
    cb_vp = cb_model.predict_proba(va_pool)[:, 1]
    cb_tp = cb_model.predict_proba(CbPool(X_test))[:, 1]
    print(f'CB   Val ROC-AUC: {roc_auc_score(y_val, cb_vp):.4f}  [{time.time()-t0:.1f}s]')

# ════════════════════════════════════════════════════════════════
# BILSTM WITH ATTENTION  (optional — GPU or CPU)
# ════════════════════════════════════════════════════════════════

bilstm_vp = bilstm_tp = None

if TORCH_AVAILABLE:
    print('\n' + '='*60)
    print(f'BILSTM + ATTENTION  ({DEVICE})')
    print('='*60)
    t0 = time.time()

    # ── vocabulary (training sequences only) ──────────────────
    _all_ops  = [op for k in train_keys for op in traces[k]]
    _op_freq  = Counter(_all_ops)
    _vocab    = {op: idx + 2 for idx, (op, _)
                 in enumerate(_op_freq.most_common(CFG['VOCAB_SIZE'] - 2))}
    _vocab['PAD'] = 0
    _vocab['UNK'] = 1
    _VSIZE = len(_vocab) + 1
    _MAXLEN = CFG['MAX_SEQ_LEN']

    def _encode_win(win: list[str]) -> list[int]:
        ids = [_vocab.get(op, 1) for op in win][:_MAXLEN]
        return ids + [0] * (_MAXLEN - len(ids))

    def _encode_keys(keys):
        seqs, ys = [], []
        for key in keys:
            for win in make_windows(traces[key], CFG['WINDOW_SIZE'], CFG['STRIDE']):
                seqs.append(_encode_win(win))
                ys.append(labels[key])
        return (
            np.array(seqs, dtype=np.int64),
            np.array(ys,   dtype=np.float32)
        )

    tr_seq, tr_y2 = _encode_keys(train_keys)
    va_seq, va_y2 = _encode_keys(val_keys)
    te_seq, te_y2 = _encode_keys(test_keys)

    # ── model ──────────────────────────────────────────────────
    class BiLSTMAttn(nn.Module):
        def __init__(self):
            super().__init__()
            E = CFG['BILSTM_EMBED']
            H = CFG['BILSTM_HIDDEN']
            D = CFG['BILSTM_DROPOUT']
            self.embed = nn.Embedding(_VSIZE, E, padding_idx=0)
            self.lstm  = nn.LSTM(
                E, H, num_layers=CFG['BILSTM_LAYERS'],
                bidirectional=True, batch_first=True, dropout=D
            )
            self.attn     = nn.Linear(H * 2, 1)
            self.layer_norm = nn.LayerNorm(H * 2)
            self.head     = nn.Sequential(
                nn.Linear(H * 2, 64),
                nn.GELU(),
                nn.Dropout(D),
                nn.Linear(64, 1),
            )

        def forward(self, x):
            mask  = (x != 0).unsqueeze(-1).float()        # (B, L, 1)
            emb   = self.embed(x)
            out, _ = self.lstm(emb)                        # (B, L, 2H)
            out   = self.layer_norm(out)
            score = self.attn(out)                         # (B, L, 1)
            score = score.masked_fill(mask == 0, -1e9)
            alpha = torch.softmax(score, dim=1)
            ctx   = (out * alpha).sum(dim=1)               # (B, 2H)
            return self.head(ctx).squeeze(-1)

    bilstm = BiLSTMAttn().to(DEVICE)

    pos_wt    = torch.tensor([scale_pos_weight], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_wt)
    opt       = torch.optim.AdamW(bilstm.parameters(), lr=1e-3, weight_decay=1e-4)
    sched     = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=CFG['BILSTM_EPOCHS'], eta_min=1e-5
    )

    _pin = (DEVICE.type == 'cuda')
    tr_loader = DataLoader(
        TensorDataset(torch.from_numpy(tr_seq), torch.from_numpy(tr_y2)),
        batch_size=CFG['BILSTM_BATCH'], shuffle=True,
        num_workers=0, pin_memory=_pin
    )

    if USE_AMP:
        _amp_scaler = GradScaler()

    best_roc   = 0.0
    best_state = None
    no_imp     = 0

    def _infer(seqs: np.ndarray) -> np.ndarray:
        bilstm.eval()
        probs = []
        BS = CFG['BILSTM_BATCH'] * 4
        with torch.no_grad():
            for i in range(0, len(seqs), BS):
                xb = torch.from_numpy(seqs[i:i+BS]).to(DEVICE)
                lgt = bilstm(xb)
                probs.append(torch.sigmoid(lgt).cpu().numpy())
        return np.concatenate(probs)

    for epoch in range(CFG['BILSTM_EPOCHS']):
        bilstm.train()
        ep_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            if USE_AMP:
                if _AMP_DEVICE:
                    with autocast(device_type=_AMP_DEVICE):
                        lgt  = bilstm(xb)
                        loss = criterion(lgt, yb)
                else:
                    with autocast():
                        lgt  = bilstm(xb)
                        loss = criterion(lgt, yb)
                _amp_scaler.scale(loss).backward()
                _amp_scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(bilstm.parameters(), 1.0)
                _amp_scaler.step(opt)
                _amp_scaler.update()
            else:
                lgt  = bilstm(xb)
                loss = criterion(lgt, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(bilstm.parameters(), 1.0)
                opt.step()
            ep_loss += loss.item()
        sched.step()

        va_pr = _infer(va_seq)
        val_roc = roc_auc_score(va_y2, va_pr)
        print(f'  Epoch {epoch+1:02d}/{CFG["BILSTM_EPOCHS"]}  '
              f'loss={ep_loss/len(tr_loader):.4f}  '
              f'val_roc={val_roc:.4f}  best={best_roc:.4f}')

        if val_roc > best_roc:
            best_roc   = val_roc
            best_state = {k: v.cpu().clone() for k, v in bilstm.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
            if no_imp >= CFG['BILSTM_PATIENCE']:
                print(f'  Early stop at epoch {epoch+1}')
                break

    bilstm.load_state_dict(best_state)
    bilstm.to(DEVICE)

    bilstm_vp = _infer(va_seq)
    bilstm_tp = _infer(te_seq)
    print(f'BiLSTM Val ROC-AUC: {best_roc:.4f}  [{time.time()-t0:.1f}s]')
    torch.save(best_state, 'bilstm_best.pt')
    joblib.dump(_vocab, 'bilstm_vocab.pkl')

# ════════════════════════════════════════════════════════════════
# ENSEMBLE STACKING  (meta-learner on val OOF preds)
# ════════════════════════════════════════════════════════════════

print('\n' + '='*60)
print('STACKING ENSEMBLE')
print('='*60)

_val_cols  = [lgb_vp, xgb_vp]
_test_cols = [lgb_tp, xgb_tp]

if cb_vp is not None:
    _val_cols.append(cb_vp);  _test_cols.append(cb_tp)

if bilstm_vp is not None:
    _val_cols.append(bilstm_vp); _test_cols.append(bilstm_tp)

val_meta  = np.column_stack(_val_cols)
test_meta = np.column_stack(_test_cols)

# Individual val ROC-AUCs
for i, name in enumerate(['LGB', 'XGB', 'CatBoost', 'BiLSTM'][:val_meta.shape[1]]):
    print(f'  {name:10s} Val ROC-AUC: {roc_auc_score(y_val, val_meta[:, i]):.4f}')

# Simple weighted average (weights learned via logistic regression on val)
meta_lr = LogisticRegression(C=5.0, max_iter=500, random_state=42)
meta_lr.fit(val_meta, y_val)
ens_vp = meta_lr.predict_proba(val_meta)[:, 1]
ens_tp = meta_lr.predict_proba(test_meta)[:, 1]

print(f'\n  Ensemble   Val ROC-AUC: {roc_auc_score(y_val, ens_vp):.4f}')
print(f'  Ensemble   Val PR-AUC : {average_precision_score(y_val, ens_vp):.4f}')
print(f'  Meta weights: {np.round(meta_lr.coef_[0], 3)}')

# ════════════════════════════════════════════════════════════════
# THRESHOLD OPTIMISATION  (maximise F1 on val)
# ════════════════════════════════════════════════════════════════

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.05, 0.95, 0.005):
    pred = (ens_vp > t).astype(int)
    s    = f1_score(y_val, pred, zero_division=0)
    if s > best_f1:
        best_f1, best_t = s, t

print(f'\n  Optimal threshold: {best_t:.3f}  (val F1={best_f1:.4f})')

# ════════════════════════════════════════════════════════════════
# FINAL EVALUATION
# ════════════════════════════════════════════════════════════════

y_prob = ens_tp
y_pred = (y_prob > best_t).astype(int)

acc  = accuracy_score(y_test,  y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test,   y_pred, zero_division=0)
f1   = f1_score(y_test,       y_pred, zero_division=0)
roc  = roc_auc_score(y_test,  y_prob)
pr   = average_precision_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp_val = cm.ravel()

print('\n' + '='*60)
print('FINAL TEST RESULTS')
print('='*60)
print(f'  Accuracy      : {acc:.4f}')
print(f'  Precision     : {prec:.4f}')
print(f'  Recall        : {rec:.4f}')
print(f'  F1 Score      : {f1:.4f}')
print(f'  ROC-AUC       : {roc:.4f}')
print(f'  PR-AUC        : {pr:.4f}')
print(f'\n  Confusion Matrix (window-level)')
print(f'    TP={tp_val}   FP={fp}')
print(f'    FN={fn}   TN={tn}')
print('='*60)

# ── Trace-level evaluation (aggregate window probs → max) ─────
print('\nTRACE-LEVEL EVALUATION (max-pool window scores)')
print('='*60)

def trace_level_eval(keys, win_probs: np.ndarray) -> None:
    """Each trace labelled anomalous if any window score > threshold."""
    idx  = 0
    tr_y, tr_p = [], []
    for key in keys:
        nw = len(make_windows(traces[key], CFG['WINDOW_SIZE'], CFG['STRIDE']))
        if bilstm_tp is not None:
            # bilstm uses same window count
            pass
        chunk = win_probs[idx: idx + nw]
        idx  += nw
        tr_y.append(labels[key])
        tr_p.append(float(np.max(chunk)))
    tr_y = np.array(tr_y)
    tr_p = np.array(tr_p)
    tr_pred = (tr_p > best_t).astype(int)
    print(f'  Accuracy  : {accuracy_score(tr_y, tr_pred):.4f}')
    print(f'  F1        : {f1_score(tr_y, tr_pred, zero_division=0):.4f}')
    print(f'  ROC-AUC   : {roc_auc_score(tr_y, tr_p):.4f}')
    print(f'  PR-AUC    : {average_precision_score(tr_y, tr_p):.4f}')

trace_level_eval(test_keys, y_prob)
print('='*60)

# ════════════════════════════════════════════════════════════════
# CHECKPOINT SAVE
# ════════════════════════════════════════════════════════════════

joblib.dump(lgb_model,   'lgb_model.pkl')
joblib.dump(xgb_model,   'xgb_model.pkl')
joblib.dump(tfidf_w,     'tfidf_word.pkl')
joblib.dump(tfidf_c,     'tfidf_char.pkl')
joblib.dump(tfidf_b,     'tfidf_bigram.pkl')
joblib.dump(scaler,      'stat_scaler.pkl')
joblib.dump(markov,      'markov_model.pkl')
joblib.dump(meta_lr,     'meta_lr.pkl')
joblib.dump({'threshold': best_t, 'window_size': CFG['WINDOW_SIZE'],
             'stride': CFG['STRIDE']}, 'pipeline_config.pkl')

if CATBOOST_AVAILABLE and cb_vp is not None:
    cb_model.save_model('catboost_model.cbm')

print('\nAll artefacts saved.')

In [ ]:
import os

print(os.listdir())

['.config', 'sample_data']


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9 — Embedding Extraction
# Updated for Hybrid-Compatible Transformer
# ════════════════════════════════════════════════════════════════

@torch.no_grad()

def extract_embeddings(model, loader):

    model.eval()

    all_embs = []

    all_labels = []

    for X, y in loader:

        X = X.to(DEVICE)

        use_amp = (
            CFG['USE_AMP']
            and
            torch.cuda.is_available()
        )

        # ------------------------------------------------
        # FORWARD
        # ------------------------------------------------

        if use_amp:

            with autocast():

                embs = model(X)

        else:

            embs = model(X)

        all_embs.append(
            embs.cpu().numpy()
        )

        all_labels.append(
            y.numpy()
        )

    return (

        np.concatenate(all_embs),

        np.concatenate(all_labels)
    )


# ════════════════════════════════════════════════════════════════
# EXTRACT ALL SPLITS
# ════════════════════════════════════════════════════════════════

embeddings = {}

for sp in ['train', 'val', 'test']:

    embs, labels = extract_embeddings(

        model,

        loaders[sp]
    )

    embeddings[sp] = (
        embs,
        labels
    )

    print(
        f'{sp}: '
        f'embeddings={embs.shape}, '
        f'labels={labels.shape}'
    )

train: embeddings=(254, 64), labels=(254,)
val: embeddings=(54, 64), labels=(54,)
test: embeddings=(56, 64), labels=(56,)


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10 — Random Forest Training
# ════════════════════════════════════════════════════════════════

X_tr, y_tr = embeddings['train']
X_val,y_val= embeddings['val']
X_te, y_te = embeddings['test']

print('Tuning Random Forest …')
best_acc, best_params, best_rf = 0.0, {}, None
param_grid = list(ParameterGrid({
    'n_estimators' : [100, 200],
    'max_features' : ['sqrt', 'log2'],
    'min_samples_leaf': [1, 2],
}))

for params in param_grid:
    rf = RandomForestClassifier(
        **params, max_depth=None,
        class_weight=CFG['RF_CLASS_WEIGHT'],
        n_jobs=-1, random_state=CFG['RANDOM_SEED'])
    rf.fit(X_tr, y_tr)
    acc = rf.score(X_val, y_val)
    if acc > best_acc:
        best_acc, best_params, best_rf = acc, params, rf

print(f'Best params: {best_params}  val_acc={best_acc:.4f}')

# Refit on train+val
X_tv = np.concatenate([X_tr, X_val])
y_tv = np.concatenate([y_tr, y_val])
best_rf.fit(X_tv, y_tv)

RF_PATH = f'{MODEL_DIR}/rf_classifier.joblib'
joblib.dump(best_rf, RF_PATH)
print(f'RF saved → {RF_PATH}')

y_pred = best_rf.predict(X_te)
y_prob = best_rf.predict_proba(X_te)[:, 1]

Tuning Random Forest …
Best params: {'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}  val_acc=0.8333
RF saved → /content/hdfs_anomaly/models/rf_classifier.joblib


In [ ]:
# ═════════IDS EVALUATION PIPELINE
# Comprehensive Metrics + Curves + Reports
# ════════════════════════════════════════════════════════════════

import json
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,

    balanced_accuracy_score,

    roc_auc_score,
    average_precision_score,

    matthews_corrcoef,
    cohen_kappa_score,

    log_loss,
    hamming_loss,
    jaccard_score,

    confusion_matrix,
    classification_report,

    roc_curve,
    precision_recall_curve
)


# ════════════════════════════════════════════════════════════════
# METRIC FUNCTION
# ════════════════════════════════════════════════════════════════

def compute_metrics(
    y_true,
    y_pred,
    y_prob
):

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    tn, fp, fn, tp = cm.ravel()

    # ------------------------------------------------
    # CORE RATES
    # ------------------------------------------------

    tpr = tp / (tp + fn + 1e-9)

    tnr = tn / (tn + fp + 1e-9)

    fpr = fp / (fp + tn + 1e-9)

    fnr = fn / (fn + tp + 1e-9)

    # ------------------------------------------------
    # RETURN
    # ------------------------------------------------

    return {

        # ------------------------------------------------
        # CLASSIFICATION
        # ------------------------------------------------

        'accuracy':

            accuracy_score(
                y_true,
                y_pred
            ),

        'precision':

            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'recall':

            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'f1':

            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        # ------------------------------------------------
        # IDS-SPECIFIC
        # ------------------------------------------------

        'specificity':

            float(tnr),

        'sensitivity':

            float(tpr),

        'balanced_accuracy':

            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        # ------------------------------------------------
        # CURVE METRICS
        # ------------------------------------------------

        'roc_auc':

            roc_auc_score(
                y_true,
                y_prob
            ),

        'pr_auc':

            average_precision_score(
                y_true,
                y_prob
            ),

        # ------------------------------------------------
        # ROBUST METRICS
        # ------------------------------------------------

        'mcc':

            matthews_corrcoef(
                y_true,
                y_pred
            ),

        'cohen_kappa':

            cohen_kappa_score(
                y_true,
                y_pred
            ),

        # ------------------------------------------------
        # LOSSES
        # ------------------------------------------------

        'log_loss':

            log_loss(
                y_true,
                y_prob
            ),

        'hamming_loss':

            hamming_loss(
                y_true,
                y_pred
            ),

        'jaccard':

            jaccard_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        # ------------------------------------------------
        # CONFUSION MATRIX
        # ------------------------------------------------

        'TP': int(tp),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),

        # ------------------------------------------------
        # RATES
        # ------------------------------------------------

        'TPR': float(tpr),
        'TNR': float(tnr),
        'FPR': float(fpr),
        'FNR': float(fnr),
    }


# ════════════════════════════════════════════════════════════════
# COMPUTE METRICS
# ════════════════════════════════════════════════════════════════

metrics = compute_metrics(
    y_te,
    y_pred,
    y_prob
)


# ════════════════════════════════════════════════════════════════
# FULL REPORT
# ════════════════════════════════════════════════════════════════

print('=' * 60)

print('     TEST SET — FULL IDS METRIC REPORT')

print('=' * 60)

for k, v in metrics.items():

    fmt = (
        f'{v:.4f}'
        if isinstance(v, float)
        else str(v)
    )

    print(f'{k:<24}: {fmt}')


# ════════════════════════════════════════════════════════════════
# CLASSIFICATION REPORT
# ════════════════════════════════════════════════════════════════

print('\n' + '=' * 60)

print('CLASSIFICATION REPORT')

print('=' * 60)

print(

    classification_report(

        y_te,

        y_pred,

        digits=4
    )
)


# ════════════════════════════════════════════════════════════════
# CONFUSION MATRIX
# ════════════════════════════════════════════════════════════════

cm = confusion_matrix(
    y_te,
    y_pred
)

print('\n' + '=' * 60)

print('CONFUSION MATRIX')

print('=' * 60)

print(cm)


# ════════════════════════════════════════════════════════════════
# ROC CURVE
# ════════════════════════════════════════════════════════════════

fpr, tpr, _ = roc_curve(
    y_te,
    y_prob
)

plt.figure(figsize=(6, 6))

plt.plot(
    fpr,
    tpr,
    label=f'AUC = {metrics["roc_auc"]:.4f}'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--'
)

plt.xlabel('False Positive Rate')

plt.ylabel('True Positive Rate')

plt.title('ROC Curve')

plt.legend()

plt.grid(True)

plt.show()


# ════════════════════════════════════════════════════════════════
# PRECISION-RECALL CURVE
# ════════════════════════════════════════════════════════════════

precision, recall, _ = precision_recall_curve(
    y_te,
    y_prob
)

plt.figure(figsize=(6, 6))

plt.plot(
    recall,
    precision,
    label=f'PR-AUC = {metrics["pr_auc"]:.4f}'
)

plt.xlabel('Recall')

plt.ylabel('Precision')

plt.title('Precision-Recall Curve')

plt.legend()

plt.grid(True)

plt.show()


# ════════════════════════════════════════════════════════════════
# SAVE METRICS
# ════════════════════════════════════════════════════════════════

metrics_path = (
    f'{RESULTS_DIR}/metrics_test.json'
)

with open(metrics_path, 'w') as f:

    json.dump(

        {

            k: (
                float(v)
                if isinstance(v, (float, np.floating))
                else int(v)
            )

            for k, v in metrics.items()

        },

        f,

        indent=2
    )

print('\nMetrics saved →', metrics_path)


# ════════════════════════════════════════════════════════════════
# SAVE PREDICTIONS
# ════════════════════════════════════════════════════════════════

preds_path = (
    f'{RESULTS_DIR}/test_predictions.npz'
)

np.savez(

    preds_path,

    y_true=y_te,

    y_pred=y_pred,

    y_prob=y_prob
)

print('Predictions saved →', preds_path)


# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════

print('\n' + '=' * 60)

print('FINAL IDS SUMMARY')

print('=' * 60)

print(f'Accuracy      : {metrics["accuracy"]:.4f}')

print(f'F1-Score      : {metrics["f1"]:.4f}')

print(f'Recall / TPR  : {metrics["recall"]:.4f}')

print(f'FNR           : {metrics["FNR"]:.4f}')

print(f'ROC-AUC       : {metrics["roc_auc"]:.4f}')

print(f'MCC           : {metrics["mcc"]:.4f}')

     TEST SET — FULL IDS METRIC REPORT
accuracy                : 0.8393
precision               : 0.8974
recall                  : 0.8750
f1                      : 0.8861
specificity             : 0.7500
sensitivity             : 0.8750
balanced_accuracy       : 0.8125
roc_auc                 : 0.9172
pr_auc                  : 0.9725
mcc                     : 0.6141
cohen_kappa             : 0.6135
log_loss                : 0.9956
hamming_loss            : 0.1607
jaccard                 : 0.7955
TP                      : 35
TN                      : 12
FP                      : 4
FN                      : 5
TPR                     : 0.8750
TNR                     : 0.7500
FPR                     : 0.2500
FNR                     : 0.1250

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.7059    0.7500    0.7273        16
           1     0.8974    0.8750    0.8861        40

    accuracy                         0.8393        56
   macro avg

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 12 — RESEARCH-GRADE VISUALIZATION PIPELINE
# ════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

from sklearn.decomposition import PCA


# ════════════════════════════════════════════════════════════════
# MATPLOTLIB STYLE
# ════════════════════════════════════════════════════════════════

plt.rcParams.update({

    'figure.dpi': 110,

    'axes.spines.top': False,

    'axes.spines.right': False
})


# ════════════════════════════════════════════════════════════════
# SAVE FUNCTION
# ════════════════════════════════════════════════════════════════

def save_fig(fig, name):

    path = f'{RESULTS_DIR}/{name}'

    fig.savefig(
        path,
        bbox_inches='tight'
    )

    plt.close(fig)

    print(f'  → {path}')


# ════════════════════════════════════════════════════════════════
# 1. TRAINING HISTORY
# ════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4)
)

# ------------------------------------------------
# LOSS
# ------------------------------------------------

axes[0].plot(
    history['tr_loss'],
    label='Train',
    lw=2
)

axes[0].plot(
    history['val_loss'],
    label='Validation',
    lw=2,
    ls='--'
)

axes[0].set_title('Loss Curves')

axes[0].set_xlabel('Epoch')

axes[0].legend()


# ------------------------------------------------
# ACCURACY
# ------------------------------------------------

axes[1].plot(
    history['tr_acc'],
    label='Train',
    lw=2
)

axes[1].plot(
    history['val_acc'],
    label='Validation',
    lw=2,
    ls='--'
)

axes[1].set_title('Accuracy Curves')

axes[1].set_xlabel('Epoch')

axes[1].legend()


# ------------------------------------------------
# LR
# ------------------------------------------------

axes[2].semilogy(
    history['lr'],
    color='orange',
    lw=2
)

axes[2].set_title('Learning Rate')

axes[2].set_xlabel('Epoch')


fig.suptitle(
    'Transformer Training History',
    fontsize=13,
    fontweight='bold'
)

fig.tight_layout()

save_fig(
    fig,
    'training_history.png'
)


# ════════════════════════════════════════════════════════════════
# 2. CONFUSION MATRIX
# ════════════════════════════════════════════════════════════════

cm = confusion_matrix(
    y_te,
    y_pred
)

fig, ax = plt.subplots(
    figsize=(5, 4)
)

im = ax.imshow(
    cm,
    cmap='Blues'
)

plt.colorbar(
    im,
    ax=ax
)

for i in range(2):

    for j in range(2):

        ax.text(

            j,
            i,

            cm[i, j],

            ha='center',

            va='center',

            fontsize=14,

            fontweight='bold',

            color=(
                'white'
                if cm[i, j] > cm.max() / 2
                else 'black'
            )
        )

ax.set_xticks([0, 1])

ax.set_yticks([0, 1])

ax.set_xticklabels([
    'Normal',
    'Anomaly'
])

ax.set_yticklabels([
    'Normal',
    'Anomaly'
])

ax.set_xlabel('Predicted')

ax.set_ylabel('True')

ax.set_title('Confusion Matrix')

fig.tight_layout()

save_fig(
    fig,
    'confusion_matrix.png'
)


# ════════════════════════════════════════════════════════════════
# 3. ROC CURVE
# ════════════════════════════════════════════════════════════════

fpr_r, tpr_r, _ = roc_curve(
    y_te,
    y_prob
)

fig, ax = plt.subplots(
    figsize=(5, 5)
)

ax.plot(

    fpr_r,

    tpr_r,

    lw=2,

    label=f'AUC={metrics["roc_auc"]:.4f}'
)

ax.plot(
    [0, 1],
    [0, 1],
    'k--',
    lw=1
)

ax.set_xlabel('False Positive Rate')

ax.set_ylabel('True Positive Rate')

ax.set_title('ROC Curve')

ax.legend()

fig.tight_layout()

save_fig(
    fig,
    'roc_curve.png'
)


# ════════════════════════════════════════════════════════════════
# 4. PRECISION-RECALL CURVE
# ════════════════════════════════════════════════════════════════

prec_r, rec_r, _ = precision_recall_curve(
    y_te,
    y_prob
)

fig, ax = plt.subplots(
    figsize=(5, 5)
)

ax.plot(

    rec_r,

    prec_r,

    lw=2,

    label=f'AP={metrics["pr_auc"]:.4f}'
)

ax.set_xlabel('Recall')

ax.set_ylabel('Precision')

ax.set_title('Precision-Recall Curve')

ax.legend()

fig.tight_layout()

save_fig(
    fig,
    'pr_curve.png'
)


# ════════════════════════════════════════════════════════════════
# 5. FEATURE IMPORTANCE
# ════════════════════════════════════════════════════════════════

imp = best_rf.feature_importances_

top_k = min(
    30,
    len(imp)
)

idx = np.argsort(imp)[::-1][:top_k]

fig, ax = plt.subplots(
    figsize=(10, 4)
)

ax.bar(
    range(top_k),
    imp[idx]
)

ax.set_xticks(
    range(top_k)
)

ax.set_xticklabels(

    [f'D{i}' for i in idx],

    rotation=45,

    ha='right',

    fontsize=7
)

ax.set_xlabel('Embedding Dimension')

ax.set_ylabel('Importance')

ax.set_title(
    f'Top-{top_k} RF Feature Importances'
)

fig.tight_layout()

save_fig(
    fig,
    'feature_importance.png'
)


# ════════════════════════════════════════════════════════════════
# 6. EMBEDDING PCA (FIXED)
# ════════════════════════════════════════════════════════════════

# IMPORTANT FIX:
# USE TRANSFORMER EMBEDDINGS

X_te_emb, _ = embeddings['test']

pca = PCA(
    n_components=2,
    random_state=42
)

proj = pca.fit_transform(
    X_te_emb
)

fig, ax = plt.subplots(
    figsize=(7, 5)
)

for lbl, clr, nm in [

    (0, '#4e8df5', 'Normal'),

    (1, '#e55353', 'Anomaly')
]:

    mask = (y_te == lbl)

    ax.scatter(

        proj[mask, 0],

        proj[mask, 1],

        c=clr,

        label=nm,

        alpha=0.6,

        s=20
    )

ax.set_title(
    'Transformer Embeddings (PCA 2D)'
)

ax.set_xlabel(

    f'PC1 '
    f'({pca.explained_variance_ratio_[0]*100:.1f}%)'
)

ax.set_ylabel(

    f'PC2 '
    f'({pca.explained_variance_ratio_[1]*100:.1f}%)'
)

ax.legend()

fig.tight_layout()

save_fig(
    fig,
    'embedding_pca.png'
)


# ════════════════════════════════════════════════════════════════
# 7. METRICS DASHBOARD
# ════════════════════════════════════════════════════════════════

dash_keys = [

    'accuracy',

    'precision',

    'recall',

    'f1',

    'roc_auc',

    'pr_auc',

    'mcc',

    'balanced_accuracy',

    'specificity',

    'sensitivity'
]

dash_vals = [
    float(metrics[k])
    for k in dash_keys
]

fig, ax = plt.subplots(
    figsize=(9, 4)
)

bars = ax.barh(
    dash_keys,
    dash_vals
)

ax.set_xlim(0, 1.1)

for bar, v in zip(
    bars,
    dash_vals
):

    ax.text(

        v + 0.01,

        bar.get_y() + bar.get_height() / 2,

        f'{v:.4f}',

        va='center',

        fontsize=9
    )

ax.set_xlabel('Score')

ax.set_title(
    'Model Performance Dashboard'
)

fig.tight_layout()

save_fig(
    fig,
    'metrics_dashboard.png'
)


# ════════════════════════════════════════════════════════════════
# COMPLETE
# ════════════════════════════════════════════════════════════════

print('\nAll research visualizations saved successfully!')

  → /content/hdfs_anomaly/results/training_history.png
  → /content/hdfs_anomaly/results/confusion_matrix.png
  → /content/hdfs_anomaly/results/roc_curve.png
  → /content/hdfs_anomaly/results/pr_curve.png
  → /content/hdfs_anomaly/results/feature_importance.png
  → /content/hdfs_anomaly/results/embedding_pca.png
  → /content/hdfs_anomaly/results/metrics_dashboard.png

All research visualizations saved successfully!


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 13 — REAL-TIME IDS INFERENCE PIPELINE
# Updated for TraceBench + Hybrid Transformer
# ════════════════════════════════════════════════════════════════

import random
import time
import numpy as np
import torch


# ════════════════════════════════════════════════════════════════
# ENCODE + PAD
# ════════════════════════════════════════════════════════════════

def encode_and_pad(
    sequence,
    event2id,
    max_len
):

    encoded = [

        event2id.get(tok, 0)

        for tok in sequence
    ]

    encoded = encoded[:max_len]

    if len(encoded) < max_len:

        encoded += [0] * (
            max_len - len(encoded)
        )

    return encoded


# ════════════════════════════════════════════════════════════════
# SINGLE PREDICTION
# ════════════════════════════════════════════════════════════════

@torch.no_grad()

def predict_single(
    sequence,
    event2id,
    model,
    rf
):

    t0 = time.perf_counter()

    # ------------------------------------------------
    # ENCODE
    # ------------------------------------------------

    ids = encode_and_pad(

        sequence,

        event2id,

        CFG['MAX_SEQ_LEN']
    )

    x = torch.tensor(

        [ids],

        dtype=torch.long

    ).to(DEVICE)

    # ------------------------------------------------
    # EMBEDDING
    # ------------------------------------------------

    use_amp = (
        CFG['USE_AMP']
        and
        torch.cuda.is_available()
    )

    if use_amp:

        with autocast():

            emb = model(x)

    else:

        emb = model(x)

    emb_np = emb.cpu().numpy()

    # ------------------------------------------------
    # RANDOM FOREST
    # ------------------------------------------------

    lbl = int(
        rf.predict(emb_np)[0]
    )

    prob = float(
        rf.predict_proba(emb_np)[0, 1]
    )

    latency = (
        time.perf_counter() - t0
    ) * 1000

    return {

        'label':

            'Anomaly'
            if lbl else
            'Normal',

        'label_id':

            lbl,

        'confidence':

            prob
            if lbl else
            1 - prob,

        'prob_anomaly':

            prob,

        'prob_normal':

            1 - prob,

        'latency_ms':

            round(latency, 3)
    }


# ════════════════════════════════════════════════════════════════
# BATCH PREDICTION
# ════════════════════════════════════════════════════════════════

@torch.no_grad()

def predict_batch(
    sequences,
    event2id,
    model,
    rf
):

    t0 = time.perf_counter()

    X = torch.tensor(

        [

            encode_and_pad(
                s,
                event2id,
                CFG['MAX_SEQ_LEN']
            )

            for s in sequences
        ],

        dtype=torch.long

    ).to(DEVICE)

    # ------------------------------------------------
    # EMBEDDINGS
    # ------------------------------------------------

    use_amp = (
        CFG['USE_AMP']
        and
        torch.cuda.is_available()
    )

    if use_amp:

        with autocast():

            embs = model(X)

    else:

        embs = model(X)

    np_embs = embs.cpu().numpy()

    # ------------------------------------------------
    # RF
    # ------------------------------------------------

    preds = rf.predict(np_embs)

    probs = rf.predict_proba(np_embs)[:, 1]

    total = (
        time.perf_counter() - t0
    ) * 1000

    results = []

    for i, (p, pr) in enumerate(

        zip(preds, probs)

    ):

        results.append({

            'index': i,

            'label':

                'Anomaly'
                if p else
                'Normal',

            'prob_anomaly':

                float(pr)
        })

    print(

        f'Batch({len(sequences)}) | '

        f'{total:.1f} ms | '

        f'{len(sequences)/(total/1000+1e-9):.0f} seq/s'
    )

    return results


# ════════════════════════════════════════════════════════════════
# REAL TRACEBENCH TOKENS
# ════════════════════════════════════════════════════════════════

event_tokens = list(
    event2id.keys()
)

print(
    '\nAvailable Operations:\n'
)

print(
    event_tokens[:20]
)


# ════════════════════════════════════════════════════════════════
# DEMO — SINGLE PREDICTION
# ════════════════════════════════════════════════════════════════

print('\n' + '─' * 60)

print('Single Predictions:\n')

for i in range(5):

    seq = [

        random.choice(event_tokens)

        for _ in range(
            random.randint(5, 20)
        )
    ]

    r = predict_single(

        seq,

        event2id,

        model,

        best_rf
    )

    print(

        f'[{i}] '

        f'label={r["label"]:8s} | '

        f'conf={r["confidence"]:.3f} | '

        f'prob_anom={r["prob_anomaly"]:.3f} | '

        f'lat={r["latency_ms"]:.2f} ms'
    )

    print(
        '  seq:',
        seq[:6]
    )


# ════════════════════════════════════════════════════════════════
# DEMO — BATCH PREDICTION
# ════════════════════════════════════════════════════════════════

print('\n' + '─' * 60)

print('Batch Prediction:\n')

batch_seqs = [

    [

        random.choice(event_tokens)

        for _ in range(15)
    ]

    for _ in range(32)
]

batch_res = predict_batch(

    batch_seqs,

    event2id,

    model,

    best_rf
)

anomalies = sum(

    1
    for r in batch_res

    if r['label'] == 'Anomaly'
)

print(

    f'\nAnomalies detected: '

    f'{anomalies}/{len(batch_res)}'
)


# ════════════════════════════════════════════════════════════════
# STREAMING SIMULATION
# ════════════════════════════════════════════════════════════════

print('\n' + '─' * 60)

print('Streaming Simulation:\n')

stream_seqs = [

    [

        random.choice(event_tokens)

        for _ in range(12)
    ]

    for _ in range(10)
]

lat_list = []

for i, seq in enumerate(stream_seqs):

    r = predict_single(

        seq,

        event2id,

        model,

        best_rf
    )

    lat_list.append(
        r['latency_ms']
    )

    print(

        f'[{i}] '

        f'{r["label"]:8s} | '

        f'prob_anom={r["prob_anomaly"]:.3f} | '

        f'lat={r["latency_ms"]:.2f} ms'
    )

print('\n' + '─' * 60)

print(

    f'Average Latency: '

    f'{np.mean(lat_list):.2f} ms'
)

print(

    f'P99 Latency: '

    f'{np.percentile(lat_list,99):.2f} ms'
)


Available Operations:

['Exception', 'OP: connect next Datanode', 'OP: new BlockReceiver', 'OP: new blockSender', 'OP: receive block', 'OP: send block', 'OP: try new BlockReader', 'RPC:abandonBlock', 'RPC:addBlock', 'RPC:commitBlockSynchronization', 'RPC:complete', 'RPC:create', 'RPC:delete', 'RPC:errorReport', 'RPC:getBlockInfo', 'RPC:getBlockLocations', 'RPC:getContentSummary', 'RPC:getFileInfo', 'RPC:getListing', 'RPC:getProtocolVersion']

────────────────────────────────────────────────────────────
Single Predictions:

[0] label=Anomaly  | conf=0.510 | prob_anom=0.510 | lat=80.05 ms
  seq: ['OP: new blockSender', 'commitBlockSynchronization', 'bestNode', 'RPC:updateBlock', 'RPC:getFileInfo', 'RPC:errorReport']
[1] label=Anomaly  | conf=0.740 | prob_anom=0.740 | lat=79.46 ms
  seq: ['OP: receive block', 'OP: new blockSender', 'RPC:create', 'RPC:startBlockRecovery', 'abandonBlock', 'recoverBlock']
[2] label=Anomaly  | conf=0.520 | prob_anom=0.520 | lat=82.35 ms
  seq: ['RPC:getProto

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 14 — REAL-TIME IDS BENCHMARKING
# Updated for TraceBench Hybrid Pipeline
# ════════════════════════════════════════════════════════════════

import time
import random
import numpy as np
import matplotlib.pyplot as plt


# ════════════════════════════════════════════════════════════════
# BENCHMARK SETTINGS
# ════════════════════════════════════════════════════════════════

N_BENCH = 500

BATCH_SZ = 64


# ════════════════════════════════════════════════════════════════
# REAL EVENT TOKENS
# ════════════════════════════════════════════════════════════════

event_tokens = list(
    event2id.keys()
)


# ════════════════════════════════════════════════════════════════
# GENERATE DUMMY STREAMS
# ════════════════════════════════════════════════════════════════

dummy_seqs = [

    [

        random.choice(event_tokens)

        for _ in range(20)
    ]

    for _ in range(N_BENCH)
]


# ════════════════════════════════════════════════════════════════
# SINGLE-SAMPLE LATENCY
# ════════════════════════════════════════════════════════════════

lat_list = []

for s in dummy_seqs[:200]:

    result = predict_single(

        s,

        event2id,

        model,

        best_rf
    )

    lat_list.append(
        result['latency_ms']
    )

lat_arr = np.array(lat_list)


# ════════════════════════════════════════════════════════════════
# BATCH THROUGHPUT
# ════════════════════════════════════════════════════════════════

t0 = time.perf_counter()

for start in range(

    0,

    N_BENCH,

    BATCH_SZ
):

    predict_batch(

        dummy_seqs[
            start:start+BATCH_SZ
        ],

        event2id,

        model,

        best_rf
    )

elapsed = (
    time.perf_counter() - t0
)

throughput = (
    N_BENCH / elapsed
)


# ════════════════════════════════════════════════════════════════
# REPORT
# ════════════════════════════════════════════════════════════════

print('\n' + '═' * 55)

print('        REAL-TIME IDS BENCHMARK REPORT')

print('═' * 55)

print(

    f'Single-sample latency (mean): '

    f'{lat_arr.mean():.3f} ms'
)

print(

    f'Single-sample latency (p50):  '

    f'{np.percentile(lat_arr, 50):.3f} ms'
)

print(

    f'Single-sample latency (p95):  '

    f'{np.percentile(lat_arr, 95):.3f} ms'
)

print(

    f'Single-sample latency (p99):  '

    f'{np.percentile(lat_arr, 99):.3f} ms'
)

print(

    f'Batch throughput:             '

    f'{throughput:.0f} seq/s'
)

print('═' * 55)


# ════════════════════════════════════════════════════════════════
# LATENCY HISTOGRAM
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(
    figsize=(7, 4)
)

ax.hist(
    lat_arr,
    bins=30,
    edgecolor='white'
)

ax.axvline(

    lat_arr.mean(),

    color='red',

    lw=1.5,

    linestyle='--',

    label=f'Mean={lat_arr.mean():.2f} ms'
)

ax.set_xlabel('Latency (ms)')

ax.set_ylabel('Count')

ax.set_title(
    'Inference Latency Distribution'
)

ax.legend()

fig.tight_layout()

hist_path = (
    f'{RESULTS_DIR}/latency_histogram.png'
)

fig.savefig(

    hist_path,

    bbox_inches='tight'
)

plt.close(fig)

print(

    '\nLatency histogram saved →',

    hist_path
)

Batch(64) | 150.0 ms | 427 seq/s
Batch(64) | 144.7 ms | 442 seq/s
Batch(64) | 139.4 ms | 459 seq/s
Batch(64) | 136.3 ms | 470 seq/s
Batch(64) | 141.0 ms | 454 seq/s
Batch(64) | 153.7 ms | 416 seq/s
Batch(64) | 139.7 ms | 458 seq/s
Batch(52) | 130.8 ms | 397 seq/s

═══════════════════════════════════════════════════════
        REAL-TIME IDS BENCHMARK REPORT
═══════════════════════════════════════════════════════
Single-sample latency (mean): 100.282 ms
Single-sample latency (p50):  84.742 ms
Single-sample latency (p95):  188.096 ms
Single-sample latency (p99):  230.586 ms
Batch throughput:             439 seq/s
═══════════════════════════════════════════════════════

Latency histogram saved → /content/hdfs_anomaly/results/latency_histogram.png


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 15 — FINAL PIPELINE SUMMARY
# Research-Grade Hybrid IDS Summary
# ════════════════════════════════════════════════════════════════

import os
import numpy as np


# ════════════════════════════════════════════════════════════════
# PARAM COUNT
# ════════════════════════════════════════════════════════════════

n_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


# ════════════════════════════════════════════════════════════════
# SUMMARY HEADER
# ════════════════════════════════════════════════════════════════

print('═' * 65)

print('   HYBRID TRANSFORMER + RANDOM FOREST IDS PIPELINE')

print('═' * 65)


# ════════════════════════════════════════════════════════════════
# MODEL INFO
# ════════════════════════════════════════════════════════════════

print('\nMODEL CONFIGURATION:\n')

print(f'  Model Parameters        : {n_params:,}')

print(f'  Vocabulary Size         : {len(event2id)}')

print(f'  Embedding Dimension     : {CFG["EMBED_DIM"]}')

print(f'  Transformer Layers      : {CFG["NUM_LAYERS"]}')

print(f'  Attention Heads         : {CFG["NUM_HEADS"]}')

print(f'  Feedforward Dimension   : {CFG["FF_DIM"]}')

print(f'  Max Sequence Length     : {CFG["MAX_SEQ_LEN"]}')

print(f'  RF Estimators           : {best_rf.n_estimators}')

print(f'  Device                  : {DEVICE}')


# ════════════════════════════════════════════════════════════════
# DATASET INFO
# ════════════════════════════════════════════════════════════════

print('\nDATASET INFORMATION:\n')

print(f'  Train Samples           : {len(y_tr)}')

print(f'  Validation Samples      : {len(y_val)}')

print(f'  Test Samples            : {len(y_te)}')

print(f'  Total Samples           : {len(y_tr)+len(y_val)+len(y_te)}')

print(f'  Test Anomalies          : {int(np.sum(y_te))}')

print(f'  Test Normals            : {int(len(y_te)-np.sum(y_te))}')


# ════════════════════════════════════════════════════════════════
# TEST METRICS
# ════════════════════════════════════════════════════════════════

print('\nTEST METRICS:\n')

metric_keys = [

    'accuracy',

    'precision',

    'recall',

    'f1',

    'roc_auc',

    'pr_auc',

    'balanced_accuracy',

    'mcc',

    'specificity',

    'sensitivity'
]

for k in metric_keys:

    print(

        f'  {k:<22}: '

        f'{metrics[k]:.4f}'
    )


# ════════════════════════════════════════════════════════════════
# CONFUSION MATRIX
# ════════════════════════════════════════════════════════════════

print('\nCONFUSION MATRIX:\n')

print(f'  TP : {metrics["TP"]}')

print(f'  TN : {metrics["TN"]}')

print(f'  FP : {metrics["FP"]}')

print(f'  FN : {metrics["FN"]}')


# ════════════════════════════════════════════════════════════════
# LATENCY INFO
# ════════════════════════════════════════════════════════════════

print('\nREAL-TIME PERFORMANCE:\n')

print(

    f'  Mean Latency (ms)       : '

    f'{lat_arr.mean():.3f}'
)

print(

    f'  P95 Latency (ms)        : '

    f'{np.percentile(lat_arr,95):.3f}'
)

print(

    f'  P99 Latency (ms)        : '

    f'{np.percentile(lat_arr,99):.3f}'
)

print(

    f'  Throughput (seq/s)      : '

    f'{throughput:.0f}'
)


# ════════════════════════════════════════════════════════════════
# PIPELINE DESCRIPTION
# ════════════════════════════════════════════════════════════════

print('\nPIPELINE ARCHITECTURE:\n')

print('  TraceBench Logs')

print('        ↓')

print('  OpName Sequence Extraction')

print('        ↓')

print('  Token Encoding + Padding')

print('        ↓')

print('  Transformer Temporal Encoder')

print('        ↓')

print('  64-D Semantic Embeddings')

print('        ↓')

print('  Random Forest Classifier')

print('        ↓')

print('  Intrusion Detection')


# ════════════════════════════════════════════════════════════════
# SAVED ARTIFACTS
# ════════════════════════════════════════════════════════════════

print('\nSAVED ARTIFACTS:\n')

for root, _, files in os.walk(BASE_DIR):

    for fn in files:

        path = os.path.join(root, fn)

        size_kb = (
            os.path.getsize(path) / 1024
        )

        rel = path.replace(
            BASE_DIR,
            '.'
        )

        print(

            f'  {rel:<50} '

            f'{size_kb:8.1f} KB'
        )


# ════════════════════════════════════════════════════════════════
# COMPLETE
# ════════════════════════════════════════════════════════════════

print('\n' + '═' * 65)

print('PIPELINE COMPLETE ✓')

print('Research-grade hybrid IDS successfully built.')

print('═' * 65)

═════════════════════════════════════════════════════════════════
   HYBRID TRANSFORMER + RANDOM FOREST IDS PIPELINE
═════════════════════════════════════════════════════════════════

MODEL CONFIGURATION:

  Model Parameters        : 76,032
  Vocabulary Size         : 74
  Embedding Dimension     : 64
  Transformer Layers      : 2
  Attention Heads         : 4
  Feedforward Dimension   : 128
  Max Sequence Length     : 100
  RF Estimators           : 100
  Device                  : cpu

DATASET INFORMATION:

  Train Samples           : 254
  Validation Samples      : 54
  Test Samples            : 56
  Total Samples           : 364
  Test Anomalies          : 40
  Test Normals            : 16

TEST METRICS:

  accuracy              : 0.8393
  precision             : 0.8974
  recall                : 0.8750
  f1                    : 0.8861
  roc_auc               : 0.9172
  pr_auc                : 0.9725
  balanced_accuracy     : 0.8125
  mcc                   : 0.6141
  specificity     